In [1]:
!lspci | grep -i nvidia

00:04.0 3D controller: NVIDIA Corporation TU104GL [Tesla T4] (rev a1)


In [2]:
!pip install torch

In [3]:
import torch
print(f"GPU 이용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"사용 중인 기기: {torch.cuda.get_device_name(0)}")

GPU 이용 가능: True
사용 중인 기기: Tesla T4


In [4]:
# 주피터 노트북 첫 번째 셀에 입력 후 실행
!pip install polars pycountry geonamescache

  Using cached pycountry-24.6.1-py3-none-any.whl.metadata (12 kB)
  Using cached geonamescache-3.0.0-py3-none-any.whl.metadata (3.3 kB)
Using cached pycountry-24.6.1-py3-none-any.whl (6.3 MB)
Using cached geonamescache-3.0.0-py3-none-any.whl (32.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [geonamescache]0m [geonamescache]


In [5]:
!pip install numpy

In [6]:
# 1. gdown 설치
!pip install gdown

# 2. 파일 다운로드 (끊김 없음)
import gdown

file_id = '1wVyqeGBiwte72lFf2JyZ3AcWOTFd3Sij' # 예: 1A2B3C...
url = f'https://drive.google.com/uc?id={file_id}'
output = 'HI-Medium_Master.parquet'

gdown.download(url, output, quiet=False)

# 3. 다시 확인 (성공하면 PAR1 나와야 함)
with open(output, "rb") as f:
    f.seek(-4, 2)
    print(f"Footer Check: {f.read(4)}") # b'PAR1' 나오면 성공!

  Using cached gdown-5.2.1-py3-none-any.whl.metadata (5.8 kB)
  Using cached PySocks-1.7.1-py3-none-any.whl.metadata (13 kB)
Using cached gdown-5.2.1-py3-none-any.whl (18 kB)
Using cached PySocks-1.7.1-py3-none-any.whl (16 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gdown]


Downloading...
From (original): https://drive.google.com/uc?id=1wVyqeGBiwte72lFf2JyZ3AcWOTFd3Sij
From (redirected): https://drive.google.com/uc?id=1wVyqeGBiwte72lFf2JyZ3AcWOTFd3Sij&confirm=t&uuid=302e129f-c450-4f1a-b8ef-f0c7cadf51d4
To: /home/tracerofjageum/my-lab/첫번째 모델/HI-Medium_Master.parquet
100%|███████████████████████████████████████████████████████████████████████| 4.01G/4.01G [00:33<00:00, 121MB/s]


Footer Check: b'PAR1'


In [7]:
import polars as pl
import re
import pycountry
import geonamescache
import numpy as np

# ---------------------------------------------------------
# A. 국가 추출 및 매핑 함수 (타 팀 로직 적용 + 오타 보정)
# ---------------------------------------------------------
def create_bank_mapping_df(unique_banks_list: list) -> pl.DataFrame:
    print(f"--- [Mapping] {len(unique_banks_list)}개 은행에 대한 국가 추출 시작 ---")
    
    gc = geonamescache.GeonamesCache()
    cities = gc.get_cities()
    city_to_country = {info["name"].upper(): info["countrycode"] for _, info in cities.items()}
    cc_to_name = {c.alpha_2: c.name for c in pycountry.countries}
    country_one_tokens = {c.name.upper() for c in pycountry.countries if len(re.findall(r"[A-Z]+", c.name.upper())) == 1}
    
    ALIASES = {
        "US": "United States", "USA": "United States", "UAE": "United Arab Emirates",
        "UK": "United Kingdom", "KOREA": "Korea, Republic of", "RUSSIA": "Russian Federation",
        "VIETNAM": "Viet Nam"
    }

    def extract_country_internal(bank_name):
        s = "" if bank_name is None else str(bank_name).upper()
        toks = re.sub(r"[^A-Z]+", " ", s).strip().split()
        if not toks: return None
        
        if len(toks) >= 2 and toks[1] == "BANK":
            x = toks[0]
            if x in ALIASES: return ALIASES[x]
            if x in country_one_tokens: return x.title()
            
        if len(toks) >= 3 and toks[0] == "BANK" and toks[1] == "OF":
            city = toks[2]
            cc = city_to_country.get(city)
            if cc: return cc_to_name.get(cc, cc)
            
        for t in toks:
            if t in ALIASES: return ALIASES[t]
            if t in country_one_tokens: return t.title()
            cc = city_to_country.get(t)
            if cc: return cc_to_name.get(cc, cc)
        return None

    # 1차 추출
    country_raw_list = [extract_country_internal(b) for b in unique_banks_list]
    
    temp_df = pl.DataFrame({"Bank_Name_Key": unique_banks_list, "Country_Raw": country_raw_list})

    # 보정 규칙
    name_norm = pl.col("Bank_Name_Key").cast(pl.Utf8).fill_null("").str.to_lowercase()
    rules = [("saudi arabia", "Saudi Arabia"), ("crypto", "Crypto"), ("crytpo", "Crypto")]
    
    rule_expr = pl.lit(None)
    for k, v in rules:
        rule_expr = pl.when(name_norm.str.contains(k, literal=True)).then(pl.lit(v)).otherwise(rule_expr)

    return temp_df.with_columns(
        pl.when(pl.col("Country_Raw").is_null())
        .then(rule_expr)
        .otherwise(pl.col("Country_Raw"))
        .fill_null("United States") # Default US
        .alias("Mapped_Country")
    ).select([
        pl.col("Bank_Name_Key").alias("Mapping_Bank_Name"),
        pl.col("Mapped_Country")
    ])

In [8]:
# ==========================================
# 0. 라이브러리 및 설정
# ==========================================
import polars as pl
import re
import pycountry
import geonamescache
import os
import gc

# 파일 경로 (GCP)
base_dir = "/home/tracerofjageum/"
file_path = base_dir + "HI-Medium_Master.parquet"
temp_path = base_dir + "temp_stacked_full.parquet" # 파일명 변경 (충돌 방지)
output_path = base_dir + "output_features_full.parquet"

# 환율 정보
exchange_rates = {
    "US Dollar": 1.0, "Bitcoin": 19800.0, "UK Pound": 1.13, "Swiss Franc": 1.02,
    "Euro": 0.99, "Canadian Dollar": 0.75, "Australian Dollar": 0.67,
    "Brazil Real": 0.19, "Saudi Riyal": 0.266, "Shekel": 0.29, "Yuan": 0.143,
    "Mexican Peso": 0.05, "Ruble": 0.0167, "Rupee": 0.0125, "Yen": 0.007
}

format_mapping = {
    "확인하다": "Cheque", "재투자": "Reinvestment", "신용카드": "Credit Card",
    "현금": "Cash", "Cheque": "Cheque", "Reinvestment": "Reinvestment",
    "Credit Card": "Credit Card", "Cash": "Cash", "Wire": "Wire", "ACH": "ACH"
}

# [함수] 국가 매핑
def create_bank_mapping_df(unique_banks_list: list) -> pl.DataFrame:
    print(f"--- [Mapping] {len(unique_banks_list)}개 은행 국가 추출 중 ---")
    gc = geonamescache.GeonamesCache()
    cities = gc.get_cities()
    city_to_country = {info["name"].upper(): info["countrycode"] for _, info in cities.items()}
    cc_to_name = {c.alpha_2: c.name for c in pycountry.countries}
    country_one_tokens = {c.name.upper() for c in pycountry.countries if len(re.findall(r"[A-Z]+", c.name.upper())) == 1}
    ALIASES = {"US": "United States", "USA": "United States", "UAE": "United Arab Emirates", "UK": "United Kingdom", "KOREA": "Korea, Republic of", "RUSSIA": "Russian Federation", "VIETNAM": "Viet Nam"}

    def extract_country_internal(bank_name):
        s = "" if bank_name is None else str(bank_name).upper()
        toks = re.sub(r"[^A-Z]+", " ", s).strip().split()
        if not toks: return None
        if len(toks) >= 2 and toks[1] == "BANK" and toks[0] in ALIASES: return ALIASES[toks[0]]
        if len(toks) >= 3 and toks[0] == "BANK" and toks[1] == "OF":
            return cc_to_name.get(city_to_country.get(toks[2]), None)
        for t in toks:
            if t in ALIASES: return ALIASES[t]
            if t in country_one_tokens: return t.title()
            if t in city_to_country: return cc_to_name.get(city_to_country[t], t)
        return None

    country_raw = [extract_country_internal(b) for b in unique_banks_list]
    temp_df = pl.DataFrame({"Bank_Name_Key": unique_banks_list, "Country_Raw": country_raw})
    
    name_norm = pl.col("Bank_Name_Key").cast(pl.Utf8).fill_null("").str.to_lowercase()
    rule_expr = pl.lit(None)
    for k, v in [("saudi arabia", "Saudi Arabia"), ("crypto", "Crypto"), ("crytpo", "Crypto")]:
        rule_expr = pl.when(name_norm.str.contains(k, literal=True)).then(pl.lit(v)).otherwise(rule_expr)
        
    return temp_df.with_columns(
        pl.when(pl.col("Country_Raw").is_null()).then(rule_expr).otherwise(pl.col("Country_Raw")).fill_null("United States").alias("Mapped_Country")
    ).select([pl.col("Bank_Name_Key").alias("Mapping_Bank_Name"), pl.col("Mapped_Country")])

# ==========================================
# Phase 1. 전처리 및 중간 저장 (Pre-calculation)
# ==========================================
print("--- Phase 1: 매핑, 풀 패키지 피처 생성, 중간 저장 ---")
try:
    # 1. 매핑 테이블
    unique_banks = pl.scan_parquet(file_path).select(pl.col("src_bank").cast(pl.String)).unique().collect().to_series().to_list()
    mapping_df = create_bank_mapping_df(unique_banks)
    q_mapping = mapping_df.lazy()
    
    # 2. Main Process (Lazy)
    q_processed = (
        pl.scan_parquet(file_path)
        .with_columns([
            pl.col("src_bank").cast(pl.String), pl.col("dst_bank").cast(pl.String),
            pl.col("from_acc").cast(pl.String), pl.col("to_acc").cast(pl.String)
        ])
        .join(q_mapping, left_on="src_bank", right_on="Mapping_Bank_Name", how="left").rename({"Mapped_Country": "From_Country"})
        .join(q_mapping, left_on="dst_bank", right_on="Mapping_Bank_Name", how="left").rename({"Mapped_Country": "To_Country"})
        .with_columns([
            pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M"),
            pl.col("Payment Format").replace_strict(format_mapping, default="Other").alias("payment_format"),
            pl.col("Payment Currency").replace_strict(exchange_rates, default=1.0).alias("rate_paid"),
            pl.col("Receiving Currency").replace_strict(exchange_rates, default=1.0).alias("rate_received"),
        ])
        .with_columns([
            (pl.col("Amount Paid") * pl.col("rate_paid")).cast(pl.Float32).alias("amt_usd"),
            (pl.col("Amount Received") * pl.col("rate_received")).cast(pl.Float32).alias("amt_rcvd_usd"),
            pl.col("from_acc").n_unique().over("src_entity").alias("Entity_Account_Count"),
            pl.when(pl.col("src_bank").str.contains("(?i)Crypto|Bit|Chain")).then(pl.lit(2))
              .when(pl.col("src_bank").str.contains("(?i)National|Indy")).then(pl.lit(1))
              .otherwise(pl.lit(0)).alias("Bank_Type_Code"),
        ])
        # --- [추가] 고급 피처 (요청하신 리스트 반영) ---
        .with_columns([
            (pl.col("amt_usd") + 1).log().alias("log_amount"),
            pl.col("Timestamp").dt.hour().alias("hour"),
            pl.col("Timestamp").dt.weekday().alias("weekday"), # 1=월, 7=일
            (pl.col("amt_usd") < 10000).cast(pl.UInt8).alias("is_small_tx"),
            ((pl.col("amt_usd") % 100) == 0).cast(pl.UInt8).alias("is_round_amt"),
            pl.col("To_Country").str.contains("(?i)Russia|Mexico|China|Korea").cast(pl.UInt8).alias("is_risk_country")
        ])
        .with_columns([
            (pl.col("hour") < 6).cast(pl.UInt8).alias("is_night"),
            (pl.col("weekday") >= 6).cast(pl.UInt8).alias("is_weekend"),
            (pl.col("amt_usd") - pl.col("amt_rcvd_usd")).alias("fee_loss"),
            (pl.col("From_Country") != pl.col("To_Country")).cast(pl.UInt8).alias("Is_Cross_Border"),
            (pl.col("src_bank") == pl.col("dst_bank")).cast(pl.UInt8).alias("Is_Intra_Bank"),
            (pl.col("payment_format").is_in(["Cheque", "Cash"])).cast(pl.UInt8).alias("Is_High_Risk_Format"),
            pl.col("Is Laundering").cast(pl.UInt8)
        ])
    )
    
    # 3. Stacking (Edge -> Node)
    # [핵심] 여기에 새로 만든 피처들이 모두 포함되어야 합니다!
    cols = [
        "Timestamp", "Is Laundering", "fee_loss", "Is_Cross_Border", 
        "Is_Intra_Bank", "Is_High_Risk_Format", "Entity_Account_Count", "Bank_Type_Code",
        "log_amount", "is_small_tx", "is_round_amt", "is_risk_country", "is_night", "is_weekend" # [추가됨]
    ]
    
    q_sender = q_processed.select([pl.col("from_acc").alias("account_id"), pl.lit(1, dtype=pl.UInt8).alias("tx_type"), pl.col("amt_usd").alias("amount"), pl.col("to_acc").alias("counterparty")] + cols)
    q_receiver = q_processed.select([pl.col("to_acc").alias("account_id"), pl.lit(0, dtype=pl.UInt8).alias("tx_type"), pl.col("amt_rcvd_usd").alias("amount"), pl.col("from_acc").alias("counterparty")] + cols)
    
    # 4. Pre-calculation & Save
    q_stacked = pl.concat([q_sender, q_receiver]).with_columns([
        pl.when(pl.col("tx_type") == 1).then(pl.col("amount")).otherwise(0.0).alias("amt_out_pre"),
        pl.when(pl.col("tx_type") == 0).then(pl.col("amount")).otherwise(0.0).alias("amt_in_pre"),
        pl.when(pl.col("tx_type") == 1).then(pl.lit(1)).otherwise(0).alias("cnt_out_pre"),
        pl.when(pl.col("tx_type") == 0).then(pl.lit(1)).otherwise(0).alias("cnt_in_pre"),
        pl.col("Timestamp").dt.truncate("1h").alias("time_group")
    ]).sort(["account_id", "Timestamp"])
    
    print(f"--- [Checkpoint] 중간 결과 저장 중... ({temp_path}) ---")
    if os.path.exists(temp_path): os.remove(temp_path)
    q_stacked.sink_parquet(temp_path)
    print("✅ 중간 저장 완료!")

except Exception as e:
    print(f"❌ Phase 1 에러: {e}")
    raise e

# ==========================================
# Phase 2. 집계 및 최종 저장 (Eager Mode)
# ==========================================
print("--- Phase 2: 집계 및 최종 피처 생성 ---")

try:
    # 1. Eager Load (리스트 타입 에러 방지)
    print("🚀 데이터 로딩 중...")
    df_eager = pl.read_parquet(temp_path)
    
    # 타입 보정
    cols_to_fix = []
    for col_name, dtype in df_eager.schema.items():
        if "List" in str(dtype):
            print(f"   🔧 List 컬럼 복구: {col_name}")
            cols_to_fix.append(pl.col(col_name).list.first().alias(col_name))
    if cols_to_fix: df_eager = df_eager.with_columns(cols_to_fix)
    
    # 안전한 캐스팅
    df_eager = df_eager.with_columns([
        pl.col("amount").cast(pl.Float32),
        pl.col("amt_out_pre").cast(pl.Float32),
        pl.col("amt_in_pre").cast(pl.Float32),
        pl.col("cnt_out_pre").cast(pl.UInt32),
        pl.col("cnt_in_pre").cast(pl.UInt32),
        pl.col("is_night").cast(pl.UInt8),
        pl.col("is_weekend").cast(pl.UInt8),
        pl.col("is_small_tx").cast(pl.UInt8),
        pl.col("is_round_amt").cast(pl.UInt8),
        pl.col("is_risk_country").cast(pl.UInt8),
        pl.col("log_amount").cast(pl.Float32)
    ])

    # 2. Aggregation (Lazy)
    print("🔄 집계 연산 시작...")
    q_features = (
        df_eager.lazy()
        .group_by(["account_id", "time_group"])
        .agg([
            pl.len().alias("cnt_1h"),
            pl.col("amount").sum().alias("sum_1h"),
            pl.col("amount").max().alias("max_1h"),
            pl.col("amount").mean().alias("avg_1h"),
            pl.col("amount").std().fill_null(0).alias("std_1h"),
            
            # Pre-calc Sum
            pl.col("cnt_out_pre").sum().alias("cnt_out_1h"),
            pl.col("cnt_in_pre").sum().alias("cnt_in_1h"),
            pl.col("amt_out_pre").sum().alias("sum_out_1h"),
            pl.col("amt_in_pre").sum().alias("sum_in_1h"),
            
            # [추가된 피처 집계] - 이제 컬럼이 있으니 에러 안 남
            pl.col("is_night").sum().alias("cnt_night"),
            pl.col("is_weekend").sum().alias("cnt_weekend"),
            pl.col("is_small_tx").sum().alias("cnt_small_tx"),
            pl.col("is_round_amt").sum().alias("cnt_round_amt"),
            pl.col("is_risk_country").sum().alias("cnt_risk_country"),
            pl.col("log_amount").mean().alias("avg_log_amount"),
            
            # Network & Risk
            pl.col("counterparty").n_unique().alias("degree_1h"),
            pl.col("Is_Cross_Border").mean().alias("ratio_cross_border"),
            pl.col("Is_High_Risk_Format").mean().alias("ratio_risk_format"),
            pl.col("Is_Intra_Bank").mean().alias("ratio_intra_bank"),
            
            # Pattern
            pl.col("Entity_Account_Count").max().alias("entity_acct_cnt"),
            pl.col("Bank_Type_Code").max().alias("bank_type"),
            
            # Target
            pl.col("Is Laundering").max().alias("is_laundering")
        ])
        .sort(["account_id", "time_group"])
    )
    
    # 3. Rolling & Ratio
    q_final = q_features.with_columns([
        pl.col("sum_1h").rolling(index_column="time_group", period="24h", closed="left").sum().over("account_id").fill_null(0).alias("sum_24h"),
        pl.col("sum_1h").rolling(index_column="time_group", period="7d", closed="left").mean().over("account_id").fill_null(0).alias("avg_7d"),
        pl.col("cnt_1h").rolling(index_column="time_group", period="24h", closed="left").sum().over("account_id").fill_null(0).alias("cnt_24h"),
    ]).with_columns([
        (pl.col("cnt_1h") / (pl.col("cnt_24h") / 24 + 1)).alias("burst_ratio"),
        (pl.col("sum_in_1h") / (pl.col("sum_out_1h") + 1)).alias("in_out_ratio"),
        (pl.col("cnt_small_tx") / (pl.col("cnt_1h") + 1)).alias("ratio_small_tx"),
        (pl.col("cnt_risk_country") / (pl.col("cnt_1h") + 1)).alias("ratio_risk_country"),
        (pl.col("cnt_night") / (pl.col("cnt_1h") + 1)).alias("ratio_night"),
        (pl.col("sum_1h") / (pl.col("avg_7d") + 1)).alias("amt_to_avg_ratio")
    ])

    # 4. 최종 저장
    print(f"--- [Final] 최종 데이터셋 저장 중... ({output_path}) ---")
    if os.path.exists(output_path): os.remove(output_path)
    
    final_df = q_final.collect()
    final_df.write_parquet(output_path)
    
    print(f"✅ 작업 완료! 파일 생성됨: {output_path}")
    print(f"   최종 Shape: {final_df.shape}")
    print("\n[Preview Columns]")
    print(final_df.columns)
    
    del df_eager, final_df
    gc.collect()
    if os.path.exists(temp_path): os.remove(temp_path)

except Exception as e:
    print(f"❌ Phase 2 에러: {e}")

--- Phase 1: 매핑, 풀 패키지 피처 생성, 중간 저장 ---
--- [Mapping] 80179개 은행 국가 추출 중 ---
--- [Checkpoint] 중간 결과 저장 중... (/home/tracerofjageum/temp_stacked_full.parquet) ---
✅ 중간 저장 완료!
--- Phase 2: 집계 및 최종 피처 생성 ---
🚀 데이터 로딩 중...
🔄 집계 연산 시작...
--- [Final] 최종 데이터셋 저장 중... (/home/tracerofjageum/output_features_full.parquet) ---
❌ Phase 2 에러: `sum` operation not supported for dtype `list[f32]`

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'sink' <---
SORT BY [col("account_id"), col("time_group")]
  AGGREGATE[maintain_order: false]
    [len().alias("cnt_1h"), col("amount").sum().alias("sum_1h"), col("amount").max().alias("max_1h"), col("amount").mean().alias("avg_1h"), col("amount").std().fill_null([0.0]).alias("std_1h"), col("cnt_out_pre").sum().alias("cnt_out_1h"), col("cnt_in_pre").sum().alias("cnt_in_1h"), col("amt_out_pre").sum().alias("sum_out_1h"), col("amt_in_pre").sum().alias("sum_in_1h"), col("is_night").sum().alias("cnt_night"), col("is_weekend").sum().alias("cnt_weekend"), col(

In [9]:
import polars as pl
import os
import gc

# 1. 경로 설정 (Phase 1은 성공했으니 그대로 둡니다)
base_dir = "/home/tracerofjageum/"
temp_path = base_dir + "temp_stacked_full.parquet"
output_path = base_dir + "output_features_full.parquet"

print("--- Phase 2: 집계 및 최종 피처 생성 (Flatten Fix) ---")

try:
    # -------------------------------------------------------
    # 1. 데이터 로드 (Eager Mode)
    # -------------------------------------------------------
    print(f"🚀 중간 데이터 로딩 중... ({temp_path})")
    df_eager = pl.read_parquet(temp_path)
    print(f"   원본 Shape: {df_eager.shape}")
    
    # -------------------------------------------------------
    # 2. [핵심] 리스트 타입 강제 평탄화 (Flatten)
    # -------------------------------------------------------
    # 데이터프레임 내의 모든 List 타입 컬럼을 찾습니다.
    list_cols = [col for col, dtype in df_eager.schema.items() if isinstance(dtype, pl.List)]
    
    if list_cols:
        print(f"⚠️ 리스트 타입 컬럼 발견! ({len(list_cols)}개) -> 스칼라로 변환합니다.")
        print(f"   대상 컬럼: {list_cols}")
        
        # 리스트 컬럼들을 한 번에 first()로 꺼내서 덮어씁니다.
        df_eager = df_eager.with_columns([
            pl.col(c).list.first().alias(c) for c in list_cols
        ])
    else:
        print("✅ 리스트 타입 컬럼이 발견되지 않았습니다. (정상)")

    # -------------------------------------------------------
    # 3. 안전한 타입 캐스팅 (Double Check)
    # -------------------------------------------------------
    # 집계에 사용될 핵심 컬럼들을 명시적으로 Float32/UInt로 변환
    df_eager = df_eager.with_columns([
        pl.col("amount").cast(pl.Float32).fill_null(0.0),
        pl.col("amt_out_pre").cast(pl.Float32).fill_null(0.0),
        pl.col("amt_in_pre").cast(pl.Float32).fill_null(0.0),
        pl.col("cnt_out_pre").cast(pl.UInt32).fill_null(0),
        pl.col("cnt_in_pre").cast(pl.UInt32).fill_null(0),
        
        # 추가 피처들
        pl.col("log_amount").cast(pl.Float32),
        pl.col("is_night").cast(pl.UInt8),
        pl.col("is_weekend").cast(pl.UInt8),
        pl.col("is_small_tx").cast(pl.UInt8),
        pl.col("is_round_amt").cast(pl.UInt8),
        pl.col("is_risk_country").cast(pl.UInt8)
    ])

    # -------------------------------------------------------
    # 4. 집계 (Aggregation)
    # -------------------------------------------------------
    print("🔄 집계 연산 시작...")
    
    q_features = (
        df_eager.lazy() # 메모리 효율을 위해 Lazy 변환
        .group_by(["account_id", "time_group"])
        .agg([
            # 1. Time & Frequency
            pl.len().alias("cnt_1h"),
            pl.col("cnt_out_pre").sum().alias("cnt_out_1h"),
            pl.col("cnt_in_pre").sum().alias("cnt_in_1h"),
            pl.col("is_night").sum().alias("cnt_night"),
            pl.col("is_weekend").sum().alias("cnt_weekend"),
            
            # 2. Amount Stats
            pl.col("amount").sum().alias("sum_1h"),
            pl.col("amount").max().alias("max_1h"),
            pl.col("amount").mean().alias("avg_1h"),
            pl.col("amount").std().fill_null(0).alias("std_1h"),
            
            pl.col("amt_out_pre").sum().alias("sum_out_1h"),
            pl.col("amt_in_pre").sum().alias("sum_in_1h"),
            pl.col("log_amount").mean().alias("avg_log_amount"),
            
            # 3. Pattern Counts
            pl.col("is_small_tx").sum().alias("cnt_small_tx"),
            pl.col("is_round_amt").sum().alias("cnt_round_amt"),
            pl.col("is_risk_country").sum().alias("cnt_risk_country"),
            pl.col("Is_High_Risk_Format").sum().alias("cnt_risk_format"),
            
            # 4. Network & Risk
            pl.col("counterparty").n_unique().alias("degree_1h"),
            pl.col("Is_Cross_Border").mean().alias("ratio_cross_border"),
            pl.col("Is_Intra_Bank").mean().alias("ratio_intra_bank"),
            
            # 5. Entity Info
            pl.col("Entity_Account_Count").max().alias("entity_acct_cnt"),
            pl.col("Bank_Type_Code").max().alias("bank_type"),
            
            # Target
            pl.col("Is Laundering").max().alias("is_laundering")
        ])
        .sort(["account_id", "time_group"])
    )
    
    # -------------------------------------------------------
    # 5. Rolling & Ratio Features
    # -------------------------------------------------------
    q_final = q_features.with_columns([
        pl.col("sum_1h").rolling(index_column="time_group", period="24h", closed="left").sum().over("account_id").fill_null(0).alias("sum_24h"),
        pl.col("sum_1h").rolling(index_column="time_group", period="7d", closed="left").mean().over("account_id").fill_null(0).alias("avg_7d"),
        pl.col("cnt_1h").rolling(index_column="time_group", period="24h", closed="left").sum().over("account_id").fill_null(0).alias("cnt_24h"),
    ]).with_columns([
        (pl.col("cnt_1h") / (pl.col("cnt_24h") / 24 + 1)).alias("burst_ratio"),
        (pl.col("sum_in_1h") / (pl.col("sum_out_1h") + 1)).alias("in_out_ratio"),
        (pl.col("cnt_small_tx") / (pl.col("cnt_1h") + 1)).alias("ratio_small_tx"),
        (pl.col("cnt_risk_country") / (pl.col("cnt_1h") + 1)).alias("ratio_risk_country"),
        (pl.col("cnt_night") / (pl.col("cnt_1h") + 1)).alias("ratio_night"),
        (pl.col("sum_1h") / (pl.col("avg_7d") + 1)).alias("amt_to_avg_ratio")
    ])

    # -------------------------------------------------------
    # 6. 최종 저장
    # -------------------------------------------------------
    print(f"--- [Final] 최종 데이터셋 저장 중... ({output_path}) ---")
    if os.path.exists(output_path): os.remove(output_path)
    
    # Collect & Save
    final_df = q_final.collect()
    final_df.write_parquet(output_path)
    
    print(f"✅ 작업 완료! 파일 생성됨: {output_path}")
    print(f"   최종 Shape: {final_df.shape}")
    print("\n[Preview Columns]")
    print(final_df.columns)
    
    # 메모리 정리
    del df_eager, final_df
    gc.collect()

except Exception as e:
    print(f"❌ Phase 2 에러 발생: {e}")
    import traceback
    traceback.print_exc()

--- Phase 2: 집계 및 최종 피처 생성 (Flatten Fix) ---
🚀 중간 데이터 로딩 중... (/home/tracerofjageum/temp_stacked_full.parquet)
   원본 Shape: (63797338, 23)
✅ 리스트 타입 컬럼이 발견되지 않았습니다. (정상)
🔄 집계 연산 시작...
--- [Final] 최종 데이터셋 저장 중... (/home/tracerofjageum/output_features_full.parquet) ---
❌ Phase 2 에러 발생: `sum` operation not supported for dtype `list[f32]`

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'sink' <---
SORT BY [col("account_id"), col("time_group")]
  AGGREGATE[maintain_order: false]
    [len().alias("cnt_1h"), col("cnt_out_pre").sum().alias("cnt_out_1h"), col("cnt_in_pre").sum().alias("cnt_in_1h"), col("is_night").sum().alias("cnt_night"), col("is_weekend").sum().alias("cnt_weekend"), col("amount").sum().alias("sum_1h"), col("amount").max().alias("max_1h"), col("amount").mean().alias("avg_1h"), col("amount").std().fill_null([0.0]).alias("std_1h"), col("amt_out_pre").sum().alias("sum_out_1h"), col("amt_in_pre").sum().alias("sum_in_1h"), col("log_amount").mean().alias("avg_log_amount"),

Traceback (most recent call last):
  File "/tmp/ipykernel_29942/2461493872.py", line 127, in <module>
    final_df = q_final.collect()
               ^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/_utils/deprecation.py", line 97, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/lazyframe/opt_flags.py", line 326, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/lazyframe/frame.py", line 2440, in collect
    return wrap_df(ldf.collect(engine, callback))
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
polars.exceptions.InvalidOperationError: `sum` operation not supported for dtype `list[f32]`

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'sink' <---
SORT BY [col("account_id"), col("time_group")]
  AGGRE

In [10]:
import polars as pl
import os
import gc

# 경로 설정
base_dir = "/home/tracerofjageum/"
temp_path = base_dir + "temp_stacked_full.parquet"
output_path = base_dir + "output_features_full.parquet"

print("--- Phase 2: Eager Mode 강제 집계 (Lazy Engine 우회) ---")

try:
    # -------------------------------------------------------
    # 1. 데이터 로드 (Eager)
    # -------------------------------------------------------
    print(f"🚀 데이터 로딩 중... ({temp_path})")
    df = pl.read_parquet(temp_path)
    print(f"   Shape: {df.shape}")

    # -------------------------------------------------------
    # 2. 타입 강제 변환 (Eager)
    # -------------------------------------------------------
    # 혹시 모를 리스트 타입을 확실하게 벗겨내고 캐스팅합니다.
    # Lazy를 쓰지 않으므로 여기서 변환하면 즉시 적용됩니다.
    print("🔧 데이터 타입 정리 중...")
    
    cols_to_cast = [
        ("amount", pl.Float32), ("amt_out_pre", pl.Float32), ("amt_in_pre", pl.Float32),
        ("cnt_out_pre", pl.UInt32), ("cnt_in_pre", pl.UInt32),
        ("log_amount", pl.Float32),
        ("is_night", pl.UInt8), ("is_weekend", pl.UInt8),
        ("is_small_tx", pl.UInt8), ("is_round_amt", pl.UInt8), ("is_risk_country", pl.UInt8),
        ("Is_High_Risk_Format", pl.UInt8), ("Is_Cross_Border", pl.UInt8), ("Is_Intra_Bank", pl.UInt8)
    ]
    
    exprs = []
    for col_name, dtype in cols_to_cast:
        # 해당 컬럼이 존재하면 캐스팅
        if col_name in df.columns:
            # 리스트면 첫번째꺼 꺼내고, 아니면 그냥 캐스팅 (안전장치)
            if df[col_name].dtype == pl.List:
                exprs.append(pl.col(col_name).list.first().cast(dtype).fill_null(0))
            else:
                exprs.append(pl.col(col_name).cast(dtype).fill_null(0))
                
    df = df.with_columns(exprs)

    # -------------------------------------------------------
    # 3. 집계 (Aggregation) - ★ Eager Mode ★
    # -------------------------------------------------------
    print("🔄 집계 연산 시작 (Eager)...")
    
    # .lazy()를 제거하고 직접 group_by 실행
    df_agg = df.group_by(["account_id", "time_group"]).agg([
        # 1. Time & Frequency
        pl.len().alias("cnt_1h"),
        pl.col("cnt_out_pre").sum().alias("cnt_out_1h"),
        pl.col("cnt_in_pre").sum().alias("cnt_in_1h"),
        pl.col("is_night").sum().alias("cnt_night"),
        pl.col("is_weekend").sum().alias("cnt_weekend"),
        
        # 2. Amount Stats
        pl.col("amount").sum().alias("sum_1h"),
        pl.col("amount").max().alias("max_1h"),
        pl.col("amount").mean().alias("avg_1h"),
        pl.col("amount").std().fill_null(0).alias("std_1h"),
        
        pl.col("amt_out_pre").sum().alias("sum_out_1h"),
        pl.col("amt_in_pre").sum().alias("sum_in_1h"),
        pl.col("log_amount").mean().alias("avg_log_amount"),
        
        # 3. Pattern Counts
        pl.col("is_small_tx").sum().alias("cnt_small_tx"),
        pl.col("is_round_amt").sum().alias("cnt_round_amt"),
        pl.col("is_risk_country").sum().alias("cnt_risk_country"),
        pl.col("Is_High_Risk_Format").sum().alias("cnt_risk_format"),
        
        # 4. Network & Risk
        pl.col("counterparty").n_unique().alias("degree_1h"),
        pl.col("Is_Cross_Border").mean().alias("ratio_cross_border"),
        pl.col("Is_Intra_Bank").mean().alias("ratio_intra_bank"),
        
        # 5. Entity Info
        pl.col("Entity_Account_Count").max().alias("entity_acct_cnt"),
        pl.col("Bank_Type_Code").max().alias("bank_type"),
        
        # Target
        pl.col("Is Laundering").max().alias("is_laundering")
    ])
    
    # 정렬 (Rolling을 위해 필수)
    df_agg = df_agg.sort(["account_id", "time_group"])
    
    # 메모리 정리
    del df
    gc.collect()

    # -------------------------------------------------------
    # 4. Rolling & Ratio Features (Eager)
    # -------------------------------------------------------
    print("🌊 Rolling Window 계산 중...")
    
    df_final = df_agg.with_columns([
        pl.col("sum_1h").rolling(index_column="time_group", period="24h", closed="left").sum().over("account_id").fill_null(0).alias("sum_24h"),
        pl.col("sum_1h").rolling(index_column="time_group", period="7d", closed="left").mean().over("account_id").fill_null(0).alias("avg_7d"),
        pl.col("cnt_1h").rolling(index_column="time_group", period="24h", closed="left").sum().over("account_id").fill_null(0).alias("cnt_24h"),
    ])
    
    df_final = df_final.with_columns([
        (pl.col("cnt_1h") / (pl.col("cnt_24h") / 24 + 1)).alias("burst_ratio"),
        (pl.col("sum_in_1h") / (pl.col("sum_out_1h") + 1)).alias("in_out_ratio"),
        (pl.col("cnt_small_tx") / (pl.col("cnt_1h") + 1)).alias("ratio_small_tx"),
        (pl.col("cnt_risk_country") / (pl.col("cnt_1h") + 1)).alias("ratio_risk_country"),
        (pl.col("cnt_night") / (pl.col("cnt_1h") + 1)).alias("ratio_night"),
        (pl.col("sum_1h") / (pl.col("avg_7d") + 1)).alias("amt_to_avg_ratio")
    ])

    # -------------------------------------------------------
    # 5. 최종 저장
    # -------------------------------------------------------
    print(f"--- [Final] 최종 데이터셋 저장 중... ({output_path}) ---")
    if os.path.exists(output_path): os.remove(output_path)
    
    df_final.write_parquet(output_path)
    
    print(f"✅ 작업 완료! 파일 생성됨: {output_path}")
    print(f"   최종 Shape: {df_final.shape}")
    print("\n[Preview Columns]")
    print(df_final.columns)
    
    # 임시 파일 삭제
    if os.path.exists(temp_path): os.remove(temp_path)

except Exception as e:
    print(f"❌ Phase 2 에러 발생: {e}")
    import traceback
    traceback.print_exc()

--- Phase 2: Eager Mode 강제 집계 (Lazy Engine 우회) ---
🚀 데이터 로딩 중... (/home/tracerofjageum/temp_stacked_full.parquet)
   Shape: (63797338, 23)
🔧 데이터 타입 정리 중...
🔄 집계 연산 시작 (Eager)...
🌊 Rolling Window 계산 중...
❌ Phase 2 에러 발생: `sum` operation not supported for dtype `list[f32]`


Traceback (most recent call last):
  File "/tmp/ipykernel_29942/2491715753.py", line 103, in <module>
    df_final = df_agg.with_columns([
               ^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/dataframe/frame.py", line 10473, in with_columns
    .collect(optimizations=QueryOptFlags._eager())
     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/_utils/deprecation.py", line 97, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/lazyframe/opt_flags.py", line 326, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/lazyframe/frame.py", line 2440, in collect
    return wrap_df(ldf.collect(engine, callback))
                   ^^^^^^^^^^^

In [11]:
import polars as pl

base_dir = "/home/tracerofjageum/"
temp_path = base_dir + "temp_stacked_full.parquet"

print(f"🧐 파일 진단 중: {temp_path}")

try:
    # 1. 파일 읽기
    df_check = pl.read_parquet(temp_path)
    
    # 2. 문제가 되는 핵심 컬럼들만 뽑아서 타입 확인
    target_cols = ["amount", "amt_out_pre", "amt_in_pre", "cnt_out_pre", "cnt_in_pre"]
    
    print("-" * 30)
    print(f"{'Column Name':<15} | {'DataType':<15} | {'First Value (Sample)'}")
    print("-" * 30)
    
    for col in target_cols:
        if col in df_check.columns:
            dtype = df_check[col].dtype
            sample = df_check[col][0] # 첫 번째 값 확인
            print(f"{col:<15} | {str(dtype):<15} | {sample}")
            
    print("-" * 30)

except Exception as e:
    print(f"❌ 진단 실패: {e}")

🧐 파일 진단 중: /home/tracerofjageum/temp_stacked_full.parquet
------------------------------
Column Name     | DataType        | First Value (Sample)
------------------------------
amount          | Float32         | 38205.01953125
amt_out_pre     | Float32         | 38205.01953125
amt_in_pre      | Float32         | 0.0
cnt_out_pre     | Int32           | 1
cnt_in_pre      | Int32           | 0
------------------------------


In [12]:
import polars as pl
import os
import gc

# 경로 설정
base_dir = "/home/tracerofjageum/"
temp_path = base_dir + "temp_stacked_full.parquet"
output_path = base_dir + "output_features_full.parquet"

print("--- Phase 2: Eager Mode 집계 (Lazy 버그 회피) ---")

try:
    # -------------------------------------------------------
    # 1. 데이터 로드 (Eager Mode)
    # -------------------------------------------------------
    # scan_parquet 대신 read_parquet을 사용하여 실제 타입을 확정합니다.
    print(f"🚀 데이터 메모리 로드 중... ({temp_path})")
    df = pl.read_parquet(temp_path)
    print(f"   Shape: {df.shape}")

    # -------------------------------------------------------
    # 2. 타입 명시적 변환 (안전장치)
    # -------------------------------------------------------
    # 진단 결과 Float32/Int32로 확인되었으나, 연산 안정성을 위해 확실하게 Cast합니다.
    print("🔧 데이터 타입 확정 중...")
    
    # 변환할 컬럼 리스트
    float_cols = ["amount", "amt_out_pre", "amt_in_pre", "log_amount"]
    uint_cols = ["cnt_out_pre", "cnt_in_pre", "is_night", "is_weekend", 
                 "is_small_tx", "is_round_amt", "is_risk_country", 
                 "Is_High_Risk_Format", "Is_Cross_Border", "Is_Intra_Bank"]
    
    # 존재하는 컬럼만 골라서 변환
    exprs = []
    for c in float_cols:
        if c in df.columns: exprs.append(pl.col(c).cast(pl.Float32).fill_null(0.0))
    for c in uint_cols:
        if c in df.columns: exprs.append(pl.col(c).cast(pl.UInt32).fill_null(0))
        
    df = df.with_columns(exprs)

    # -------------------------------------------------------
    # 3. 집계 (Aggregation)
    # -------------------------------------------------------
    print("🔄 GroupBy 집계 시작...")
    
    # Eager DataFrame에서 바로 group_by 수행
    df_agg = df.group_by(["account_id", "time_group"]).agg([
        # 1. Basic Stats
        pl.len().alias("cnt_1h"),
        pl.col("amount").sum().alias("sum_1h"),
        pl.col("amount").max().alias("max_1h"),
        pl.col("amount").mean().alias("avg_1h"),
        pl.col("amount").std().fill_null(0).alias("std_1h"),
        
        # 2. Pre-calculated Sums (이제 에러 날 수 없음)
        pl.col("cnt_out_pre").sum().alias("cnt_out_1h"),
        pl.col("cnt_in_pre").sum().alias("cnt_in_1h"),
        pl.col("amt_out_pre").sum().alias("sum_out_1h"),
        pl.col("amt_in_pre").sum().alias("sum_in_1h"),
        
        # 3. Feature Counts
        pl.col("is_night").sum().alias("cnt_night"),
        pl.col("is_weekend").sum().alias("cnt_weekend"),
        pl.col("is_small_tx").sum().alias("cnt_small_tx"),
        pl.col("is_round_amt").sum().alias("cnt_round_amt"),
        pl.col("is_risk_country").sum().alias("cnt_risk_country"),
        pl.col("Is_High_Risk_Format").sum().alias("cnt_risk_format"),
        
        # 4. Averages / Max
        pl.col("log_amount").mean().alias("avg_log_amount"),
        pl.col("counterparty").n_unique().alias("degree_1h"),
        pl.col("Is_Cross_Border").mean().alias("ratio_cross_border"),
        pl.col("Is_Intra_Bank").mean().alias("ratio_intra_bank"),
        
        pl.col("Entity_Account_Count").max().alias("entity_acct_cnt"),
        pl.col("Bank_Type_Code").max().alias("bank_type"),
        
        # Target
        pl.col("Is Laundering").max().alias("is_laundering")
    ])
    
    # 정렬 (Rolling을 위해 필수)
    df_agg = df_agg.sort(["account_id", "time_group"])
    
    # 메모리 정리 (원본 df 삭제)
    del df
    gc.collect()

    # -------------------------------------------------------
    # 4. Rolling Window & Ratios
    # -------------------------------------------------------
    print("🌊 Rolling Window & 파생 변수 계산...")
    
    df_final = df_agg.with_columns([
        pl.col("sum_1h").rolling(index_column="time_group", period="24h", closed="left").sum().over("account_id").fill_null(0).alias("sum_24h"),
        pl.col("sum_1h").rolling(index_column="time_group", period="7d", closed="left").mean().over("account_id").fill_null(0).alias("avg_7d"),
        pl.col("cnt_1h").rolling(index_column="time_group", period="24h", closed="left").sum().over("account_id").fill_null(0).alias("cnt_24h"),
    ])
    
    df_final = df_final.with_columns([
        (pl.col("cnt_1h") / (pl.col("cnt_24h") / 24 + 1)).alias("burst_ratio"),
        (pl.col("sum_in_1h") / (pl.col("sum_out_1h") + 1)).alias("in_out_ratio"),
        (pl.col("cnt_small_tx") / (pl.col("cnt_1h") + 1)).alias("ratio_small_tx"),
        (pl.col("cnt_risk_country") / (pl.col("cnt_1h") + 1)).alias("ratio_risk_country"),
        (pl.col("cnt_night") / (pl.col("cnt_1h") + 1)).alias("ratio_night"),
        (pl.col("sum_1h") / (pl.col("avg_7d") + 1)).alias("amt_to_avg_ratio")
    ])

    # -------------------------------------------------------
    # 5. 최종 저장
    # -------------------------------------------------------
    print(f"--- [Final] 최종 데이터셋 저장 중... ({output_path}) ---")
    if os.path.exists(output_path): os.remove(output_path)
    
    df_final.write_parquet(output_path)
    
    print(f"✅ 작업 완료! 파일 생성됨: {output_path}")
    print(f"   최종 Shape: {df_final.shape}")
    
    # 미리보기
    print("\n[Preview Features]")
    print(df_final.columns)
    print(df_final.head())

except Exception as e:
    print(f"❌ 에러 발생: {e}")
    import traceback
    traceback.print_exc()

--- Phase 2: Eager Mode 집계 (Lazy 버그 회피) ---
🚀 데이터 메모리 로드 중... (/home/tracerofjageum/temp_stacked_full.parquet)
   Shape: (63797338, 23)
🔧 데이터 타입 확정 중...
🔄 GroupBy 집계 시작...
🌊 Rolling Window & 파생 변수 계산...
❌ 에러 발생: `sum` operation not supported for dtype `list[f32]`


Traceback (most recent call last):
  File "/tmp/ipykernel_29942/162142630.py", line 95, in <module>
    df_final = df_agg.with_columns([
               ^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/dataframe/frame.py", line 10473, in with_columns
    .collect(optimizations=QueryOptFlags._eager())
     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/_utils/deprecation.py", line 97, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/lazyframe/opt_flags.py", line 326, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/lazyframe/frame.py", line 2440, in collect
    return wrap_df(ldf.collect(engine, callback))
                   ^^^^^^^^^^^^^

In [13]:
import polars as pl
import os
import gc

# 경로 설정 (Phase 1은 성공했으므로 그대로 둡니다)
base_dir = "/home/tracerofjageum/"
temp_path = base_dir + "temp_stacked_full.parquet"
output_path = base_dir + "output_features_full.parquet"

print("--- Phase 2: 집계 및 Rolling (Float64 안정화 버전) ---")

try:
    # -------------------------------------------------------
    # 1. 데이터 로드 (Eager)
    # -------------------------------------------------------
    print(f"🚀 중간 데이터 로딩 중... ({temp_path})")
    df = pl.read_parquet(temp_path)
    
    # -------------------------------------------------------
    # 2. 타입 변환 (Float32 -> Float64)
    # -------------------------------------------------------
    # ★ 핵심 수정: Float32는 Rolling에서 가끔 불안정하므로 Float64로 승격합니다.
    print("🔧 연산 안정성을 위해 Float64로 변환 중...")
    
    df = df.with_columns([
        pl.col("amount").cast(pl.Float64).fill_null(0.0),
        pl.col("amt_out_pre").cast(pl.Float64).fill_null(0.0),
        pl.col("amt_in_pre").cast(pl.Float64).fill_null(0.0),
        pl.col("log_amount").cast(pl.Float64).fill_null(0.0),
        # 카운트도 넉넉하게 Int64로
        pl.col("cnt_out_pre").cast(pl.UInt64).fill_null(0),
        pl.col("cnt_in_pre").cast(pl.UInt64).fill_null(0),
    ])

    # -------------------------------------------------------
    # 3. 1차 집계 (Aggregation)
    # -------------------------------------------------------
    print("🔄 GroupBy 집계 (1h 단위)...")
    
    # 여기서 리스트가 생길 일은 없습니다.
    df_agg = df.group_by(["account_id", "time_group"]).agg([
        # Basic Stats
        pl.len().cast(pl.UInt64).alias("cnt_1h"),
        pl.col("amount").sum().alias("sum_1h"),
        pl.col("amount").max().alias("max_1h"),
        pl.col("amount").mean().alias("avg_1h"),
        pl.col("amount").std().fill_null(0).alias("std_1h"),
        
        # Pre-calc Sums
        pl.col("cnt_out_pre").sum().alias("cnt_out_1h"),
        pl.col("cnt_in_pre").sum().alias("cnt_in_1h"),
        pl.col("amt_out_pre").sum().alias("sum_out_1h"),
        pl.col("amt_in_pre").sum().alias("sum_in_1h"),
        
        # Counts
        pl.col("is_night").sum().cast(pl.UInt64).alias("cnt_night"),
        pl.col("is_weekend").sum().cast(pl.UInt64).alias("cnt_weekend"),
        pl.col("is_small_tx").sum().cast(pl.UInt64).alias("cnt_small_tx"),
        pl.col("is_round_amt").sum().cast(pl.UInt64).alias("cnt_round_amt"),
        pl.col("is_risk_country").sum().cast(pl.UInt64).alias("cnt_risk_country"),
        pl.col("Is_High_Risk_Format").sum().cast(pl.UInt64).alias("cnt_risk_format"),
        
        # Averages
        pl.col("log_amount").mean().alias("avg_log_amount"),
        pl.col("Is_Cross_Border").mean().alias("ratio_cross_border"),
        pl.col("Is_Intra_Bank").mean().alias("ratio_intra_bank"),
        
        # Max
        pl.col("counterparty").n_unique().cast(pl.UInt64).alias("degree_1h"),
        pl.col("Entity_Account_Count").max().cast(pl.UInt64).alias("entity_acct_cnt"),
        pl.col("Bank_Type_Code").max().cast(pl.UInt8).alias("bank_type"),
        pl.col("Is Laundering").max().cast(pl.UInt8).alias("is_laundering")
    ])
    
    # 메모리 정리
    del df
    gc.collect()
    
    # 정렬 및 검증
    print("📐 정렬 및 스키마 검증...")
    df_agg = df_agg.sort(["account_id", "time_group"])
    
    # 여기서 스키마 한 번 찍어봅니다 (디버깅용)
    print("   Aggregated Schema Check:")
    for col, dtype in df_agg.schema.items():
        if "List" in str(dtype):
            print(f"   🚨 경고: {col} 이 List 타입입니다!")
    
    # -------------------------------------------------------
    # 4. Rolling Window (Rolling 문법 변경)
    # -------------------------------------------------------
    print("🌊 Rolling Window 계산 중 (by parameter 사용)...")
    
    # ★ 핵심 수정: .over() 대신 .rolling(..., by="account_id") 사용
    # Polars 최신 버전에서는 이 방식이 훨씬 안정적입니다.
    # 만약 구버전이라 'by' 에러가 나면 except로 빠지도록 처리했습니다.

    try:
        df_final = df_agg.with_columns([
            pl.col("sum_1h").rolling(index_column="time_group", period="24h", by="account_id", closed="left").sum().fill_null(0.0).alias("sum_24h"),
            pl.col("sum_1h").rolling(index_column="time_group", period="7d", by="account_id", closed="left").mean().fill_null(0.0).alias("avg_7d"),
            pl.col("cnt_1h").rolling(index_column="time_group", period="24h", by="account_id", closed="left").sum().fill_null(0).alias("cnt_24h"),
        ])
    except Exception as e_rolling:
        print(f"⚠️ 'by' 파라미터 실패 ({e_rolling}), 기존 방식(.over) 재시도 (Float64라 성공할 것)...")
        df_final = df_agg.with_columns([
            pl.col("sum_1h").rolling(index_column="time_group", period="24h", closed="left").sum().over("account_id").fill_null(0.0).alias("sum_24h"),
            pl.col("sum_1h").rolling(index_column="time_group", period="7d", closed="left").mean().over("account_id").fill_null(0.0).alias("avg_7d"),
            pl.col("cnt_1h").rolling(index_column="time_group", period="24h", closed="left").sum().over("account_id").fill_null(0).alias("cnt_24h"),
        ])

    # -------------------------------------------------------
    # 5. Ratio Features & Save
    # -------------------------------------------------------
    print("➗ 파생 변수 계산 중...")
    df_final = df_final.with_columns([
        (pl.col("cnt_1h") / (pl.col("cnt_24h") / 24 + 1)).alias("burst_ratio"),
        (pl.col("sum_in_1h") / (pl.col("sum_out_1h") + 1)).alias("in_out_ratio"),
        (pl.col("cnt_small_tx") / (pl.col("cnt_1h") + 1)).alias("ratio_small_tx"),
        (pl.col("cnt_risk_country") / (pl.col("cnt_1h") + 1)).alias("ratio_risk_country"),
        (pl.col("cnt_night") / (pl.col("cnt_1h") + 1)).alias("ratio_night"),
        (pl.col("sum_1h") / (pl.col("avg_7d") + 1)).alias("amt_to_avg_ratio")
    ])

    print(f"--- [Final] 최종 데이터셋 저장 중... ({output_path}) ---")
    if os.path.exists(output_path): os.remove(output_path)
    
    df_final.write_parquet(output_path)
    
    print(f"✅ 작업 완료! 파일 생성됨: {output_path}")
    print(f"   최종 Shape: {df_final.shape}")
    print("\n[Preview Columns]")
    print(df_final.columns)
    
    # 임시 파일 삭제
    if os.path.exists(temp_path): os.remove(temp_path)

except Exception as e:
    print(f"❌ 에러 발생: {e}")
    import traceback
    traceback.print_exc()

--- Phase 2: 집계 및 Rolling (Float64 안정화 버전) ---
🚀 중간 데이터 로딩 중... (/home/tracerofjageum/temp_stacked_full.parquet)
🔧 연산 안정성을 위해 Float64로 변환 중...
🔄 GroupBy 집계 (1h 단위)...
📐 정렬 및 스키마 검증...
   Aggregated Schema Check:
🌊 Rolling Window 계산 중 (by parameter 사용)...
⚠️ 'by' 파라미터 실패 (Expr.rolling() got an unexpected keyword argument 'by'), 기존 방식(.over) 재시도 (Float64라 성공할 것)...
❌ 에러 발생: `sum` operation not supported for dtype `list[f64]`


Traceback (most recent call last):
  File "/tmp/ipykernel_29942/2831093221.py", line 100, in <module>
    pl.col("sum_1h").rolling(index_column="time_group", period="24h", by="account_id", closed="left").sum().fill_null(0.0).alias("sum_24h"),
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: Expr.rolling() got an unexpected keyword argument 'by'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/tmp/ipykernel_29942/2831093221.py", line 106, in <module>
    df_final = df_agg.with_columns([
               ^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/dataframe/frame.py", line 10473, in with_columns
    .collect(optimizations=QueryOptFlags._eager())
     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/_utils/deprecation.py", line 97, in 

In [14]:
# 1. DuckDB 설치 (이미 있으면 넘어갑니다)
!pip install duckdb

import polars as pl
import duckdb
import os
import gc

# 경로 설정
base_dir = "/home/tracerofjageum/"
temp_path = base_dir + "temp_stacked_full.parquet"
output_path = base_dir + "output_features_full.parquet"

print("--- Phase 2: Polars + DuckDB 하이브리드 집계 (Rolling 에러 해결) ---")

try:
    # -------------------------------------------------------
    # 1. Polars로 데이터 로드 및 1차 집계 (이건 에러 안 남)
    # -------------------------------------------------------
    print(f"🚀 중간 데이터 로드 중... ({temp_path})")
    df = pl.read_parquet(temp_path)
    
    # 안전한 타입 변환
    df = df.with_columns([
        pl.col("amount").cast(pl.Float64).fill_null(0.0),
        pl.col("amt_out_pre").cast(pl.Float64).fill_null(0.0),
        pl.col("amt_in_pre").cast(pl.Float64).fill_null(0.0),
        pl.col("log_amount").cast(pl.Float64).fill_null(0.0),
        pl.col("cnt_out_pre").cast(pl.Int64).fill_null(0),
        pl.col("cnt_in_pre").cast(pl.Int64).fill_null(0),
        pl.col("is_night").cast(pl.Int64),
        pl.col("is_weekend").cast(pl.Int64),
        pl.col("is_small_tx").cast(pl.Int64),
        pl.col("is_risk_country").cast(pl.Int64),
    ])

    print("🔄 1차 집계 (GroupBy) 수행 중 (Polars)...")
    # Rolling 없이 단순 집계만 수행 (매우 빠르고 안정적)
    df_agg = df.group_by(["account_id", "time_group"]).agg([
        # Basic Stats
        pl.len().cast(pl.Int64).alias("cnt_1h"),
        pl.col("amount").sum().alias("sum_1h"),
        pl.col("amount").max().alias("max_1h"),
        pl.col("amount").mean().alias("avg_1h"),
        pl.col("amount").std().fill_null(0).alias("std_1h"),
        
        # Pre-calc Sums
        pl.col("cnt_out_pre").sum().alias("cnt_out_1h"),
        pl.col("cnt_in_pre").sum().alias("cnt_in_1h"),
        pl.col("amt_out_pre").sum().alias("sum_out_1h"),
        pl.col("amt_in_pre").sum().alias("sum_in_1h"),
        
        # Counts
        pl.col("is_night").sum().alias("cnt_night"),
        pl.col("is_weekend").sum().alias("cnt_weekend"),
        pl.col("is_small_tx").sum().alias("cnt_small_tx"),
        pl.col("is_round_amt").sum().alias("cnt_round_amt"),
        pl.col("is_risk_country").sum().alias("cnt_risk_country"),
        pl.col("Is_High_Risk_Format").sum().alias("cnt_risk_format"),
        
        # Averages / Max
        pl.col("log_amount").mean().alias("avg_log_amount"),
        pl.col("Is_Cross_Border").mean().alias("ratio_cross_border"),
        pl.col("Is_Intra_Bank").mean().alias("ratio_intra_bank"),
        pl.col("counterparty").n_unique().alias("degree_1h"),
        pl.col("Entity_Account_Count").max().alias("entity_acct_cnt"),
        pl.col("Bank_Type_Code").max().alias("bank_type"),
        pl.col("Is Laundering").max().alias("is_laundering")
    ]).sort(["account_id", "time_group"])

    print(f"   집계 완료 Shape: {df_agg.shape}")
    
    # 메모리 정리
    del df
    gc.collect()

    # -------------------------------------------------------
    # 2. DuckDB로 Rolling Window 계산 (★핵심 치트키)
    # -------------------------------------------------------
    print("🌊 DuckDB로 Rolling Window 계산 중 (SQL)...")
    
    # DuckDB는 Polars DataFrame(df_agg)을 바로 쿼리할 수 있습니다.
    # RANGE BETWEEN ... AND INTERVAL '1 HOUR' PRECEDING -> 현재 1시간을 제외한 과거 24시간
    
    query = """
    SELECT 
        *,
        -- 24시간 합계 (현재 시간 제외: 25시간 전 ~ 1시간 전)
        SUM(sum_1h) OVER (
            PARTITION BY account_id 
            ORDER BY time_group 
            RANGE BETWEEN INTERVAL '24 HOURS' PRECEDING AND INTERVAL '1 HOUR' PRECEDING
        ) as sum_24h,
        
        -- 24시간 횟수
        SUM(cnt_1h) OVER (
            PARTITION BY account_id 
            ORDER BY time_group 
            RANGE BETWEEN INTERVAL '24 HOURS' PRECEDING AND INTERVAL '1 HOUR' PRECEDING
        ) as cnt_24h,
        
        -- 7일 평균
        AVG(sum_1h) OVER (
            PARTITION BY account_id 
            ORDER BY time_group 
            RANGE BETWEEN INTERVAL '7 DAYS' PRECEDING AND INTERVAL '1 HOUR' PRECEDING
        ) as avg_7d
        
    FROM df_agg
    """
    
    # 쿼리 실행 및 Polars로 다시 변환
    df_rolling = duckdb.sql(query).pl()
    
    # NULL 값 처리 (윈도우 초기값)
    df_rolling = df_rolling.with_columns([
        pl.col("sum_24h").fill_null(0.0),
        pl.col("cnt_24h").fill_null(0),
        pl.col("avg_7d").fill_null(0.0)
    ])

    # -------------------------------------------------------
    # 3. 파생 변수 계산 및 저장
    # -------------------------------------------------------
    print("➗ 파생 변수 계산 중...")
    
    df_final = df_rolling.with_columns([
        (pl.col("cnt_1h") / (pl.col("cnt_24h") / 24 + 1)).alias("burst_ratio"),
        (pl.col("sum_in_1h") / (pl.col("sum_out_1h") + 1)).alias("in_out_ratio"),
        (pl.col("cnt_small_tx") / (pl.col("cnt_1h") + 1)).alias("ratio_small_tx"),
        (pl.col("cnt_risk_country") / (pl.col("cnt_1h") + 1)).alias("ratio_risk_country"),
        (pl.col("cnt_night") / (pl.col("cnt_1h") + 1)).alias("ratio_night"),
        (pl.col("sum_1h") / (pl.col("avg_7d") + 1)).alias("amt_to_avg_ratio")
    ])

    print(f"--- [Final] 최종 데이터셋 저장 중... ({output_path}) ---")
    if os.path.exists(output_path): os.remove(output_path)
    
    df_final.write_parquet(output_path)
    
    print(f"✅ 작업 완료! 파일 생성됨: {output_path}")
    print(f"   최종 Shape: {df_final.shape}")
    print("\n[Preview Columns]")
    print(df_final.columns)
    
    # 임시 파일 삭제
    if os.path.exists(temp_path): os.remove(temp_path)

except Exception as e:
    print(f"❌ 에러 발생: {e}")
    import traceback
    traceback.print_exc()

  Using cached duckdb-1.4.4-cp312-cp312-manylinux_2_26_x86_64.manylinux_2_28_x86_64.whl.metadata (4.3 kB)
Using cached duckdb-1.4.4-cp312-cp312-manylinux_2_26_x86_64.manylinux_2_28_x86_64.whl (20.4 MB)
--- Phase 2: Polars + DuckDB 하이브리드 집계 (Rolling 에러 해결) ---
🚀 중간 데이터 로드 중... (/home/tracerofjageum/temp_stacked_full.parquet)
🔄 1차 집계 (GroupBy) 수행 중 (Polars)...
   집계 완료 Shape: (28756304, 24)
🌊 DuckDB로 Rolling Window 계산 중 (SQL)...
➗ 파생 변수 계산 중...
--- [Final] 최종 데이터셋 저장 중... (/home/tracerofjageum/output_features_full.parquet) ---
✅ 작업 완료! 파일 생성됨: /home/tracerofjageum/output_features_full.parquet
   최종 Shape: (28756304, 33)

[Preview Columns]
['account_id', 'time_group', 'cnt_1h', 'sum_1h', 'max_1h', 'avg_1h', 'std_1h', 'cnt_out_1h', 'cnt_in_1h', 'sum_out_1h', 'sum_in_1h', 'cnt_night', 'cnt_weekend', 'cnt_small_tx', 'cnt_round_amt', 'cnt_risk_country', 'cnt_risk_format', 'avg_log_amount', 'ratio_cross_border', 'ratio_intra_bank', 'degree_1h', 'entity_acct_cnt', 'bank_type', 'is_laundering', 

In [15]:
# 1. 필수 라이브러리 설치
!pip install pyarrow

import polars as pl
import duckdb
import os
import gc

# 경로 설정
base_dir = "/home/tracerofjageum/"
temp_path = base_dir + "temp_stacked_full.parquet"
output_path = base_dir + "output_features_full.parquet"

print("--- Phase 2: Polars + DuckDB + PyArrow (에러 원천 차단 버전) ---")

try:
    # -------------------------------------------------------
    # 1. Polars로 데이터 로드 및 1차 집계 (Eager)
    # -------------------------------------------------------
    print(f"🚀 중간 데이터 로드 중... ({temp_path})")
    df = pl.read_parquet(temp_path)
    
    # 타입 정규화 (DuckDB 연산 안정성 확보)
    df = df.with_columns([
        pl.col("amount").cast(pl.Float64).fill_null(0.0),
        pl.col("amt_out_pre").cast(pl.Float64).fill_null(0.0),
        pl.col("amt_in_pre").cast(pl.Float64).fill_null(0.0),
        pl.col("log_amount").cast(pl.Float64).fill_null(0.0),
        pl.col("cnt_out_pre").cast(pl.Int64),
        pl.col("cnt_in_pre").cast(pl.Int64),
        pl.col("is_night").cast(pl.Int64),
        pl.col("is_weekend").cast(pl.Int64),
        pl.col("is_small_tx").cast(pl.Int64),
        pl.col("is_risk_country").cast(pl.Int64),
    ])

    print("🔄 1차 집계 (GroupBy) 수행 중...")
    df_agg = df.group_by(["account_id", "time_group"]).agg([
        pl.len().cast(pl.Int64).alias("cnt_1h"),
        pl.col("amount").sum().alias("sum_1h"),
        pl.col("amount").max().alias("max_1h"),
        pl.col("amount").mean().alias("avg_1h"),
        pl.col("amount").std().fill_null(0).alias("std_1h"),
        pl.col("cnt_out_pre").sum().alias("cnt_out_1h"),
        pl.col("cnt_in_pre").sum().alias("cnt_in_1h"),
        pl.col("amt_out_pre").sum().alias("sum_out_1h"),
        pl.col("amt_in_pre").sum().alias("sum_in_1h"),
        pl.col("is_night").sum().alias("cnt_night"),
        pl.col("is_weekend").sum().alias("cnt_weekend"),
        pl.col("is_small_tx").sum().alias("cnt_small_tx"),
        pl.col("is_round_amt").sum().alias("cnt_round_amt"),
        pl.col("is_risk_country").sum().alias("cnt_risk_country"),
        pl.col("Is_High_Risk_Format").sum().alias("cnt_risk_format"),
        pl.col("log_amount").mean().alias("avg_log_amount"),
        pl.col("Is_Cross_Border").mean().alias("ratio_cross_border"),
        pl.col("Is_Intra_Bank").mean().alias("ratio_intra_bank"),
        pl.col("counterparty").n_unique().alias("degree_1h"),
        pl.col("Entity_Account_Count").max().alias("entity_acct_cnt"),
        pl.col("Bank_Type_Code").max().alias("bank_type"),
        pl.col("Is Laundering").max().alias("is_laundering")
    ]).sort(["account_id", "time_group"])

    # -------------------------------------------------------
    # 2. DuckDB로 Rolling Window 계산 (버그 프리)
    # -------------------------------------------------------
    print("🌊 DuckDB SQL을 이용한 Rolling Window 계산...")
    
    # 윈도우 함수 정의: 1시간 전을 기준으로 과거 24시간 및 7일 데이터 합산
    query = """
    SELECT 
        *,
        SUM(sum_1h) OVER (
            PARTITION BY account_id 
            ORDER BY time_group 
            RANGE BETWEEN INTERVAL '24 HOURS' PRECEDING AND INTERVAL '1 HOUR' PRECEDING
        ) as sum_24h,
        SUM(cnt_1h) OVER (
            PARTITION BY account_id 
            ORDER BY time_group 
            RANGE BETWEEN INTERVAL '24 HOURS' PRECEDING AND INTERVAL '1 HOUR' PRECEDING
        ) as cnt_24h,
        AVG(sum_1h) OVER (
            PARTITION BY account_id 
            ORDER BY time_group 
            RANGE BETWEEN INTERVAL '7 DAYS' PRECEDING AND INTERVAL '1 HOUR' PRECEDING
        ) as avg_7d
    FROM df_agg
    """
    
    # DuckDB 실행 결과를 Polars 데이터프레임으로 수신
    df_final = duckdb.sql(query).pl()

    # -------------------------------------------------------
    # 3. 최종 비율 계산 및 저장
    # -------------------------------------------------------
    print("➗ 패턴 비율(Ratio) 피처 계산 중...")
    
    df_final = df_final.with_columns([
        pl.col("sum_24h").fill_null(0.0),
        pl.col("cnt_24h").fill_null(0),
        pl.col("avg_7d").fill_null(0.0)
    ]).with_columns([
        (pl.col("cnt_1h") / (pl.col("cnt_24h") / 24 + 1)).alias("burst_ratio"),
        (pl.col("sum_in_1h") / (pl.col("sum_out_1h") + 1)).alias("in_out_ratio"),
        (pl.col("cnt_small_tx") / (pl.col("cnt_1h") + 1)).alias("ratio_small_tx"),
        (pl.col("cnt_risk_country") / (pl.col("cnt_1h") + 1)).alias("ratio_risk_country"),
        (pl.col("cnt_night") / (pl.col("cnt_1h") + 1)).alias("ratio_night"),
        (pl.col("sum_1h") / (pl.col("avg_7d") + 1)).alias("amt_to_avg_ratio")
    ])

    print(f"--- [Final] 결과 저장 중: {output_path} ---")
    if os.path.exists(output_path): os.remove(output_path)
    
    df_final.write_parquet(output_path)
    
    print(f"✅ 축하합니다! 모든 전처리가 완료되었습니다.")
    print(f"   최종 Shape: {df_final.shape}")
    
    # 임시 파일 정리
    if os.path.exists(temp_path): os.remove(temp_path)
    
    # 미리보기
    print("\n[Feature Preview]")
    print(df_final.head())

except Exception as e:
    print(f"❌ 에러 발생: {e}")
    import traceback
    traceback.print_exc()

--- Phase 2: Polars + DuckDB + PyArrow (에러 원천 차단 버전) ---
🚀 중간 데이터 로드 중... (/home/tracerofjageum/temp_stacked_full.parquet)
❌ 에러 발생: No such file or directory (os error 2): /home/tracerofjageum/temp_stacked_full.parquet


Traceback (most recent call last):
  File "/tmp/ipykernel_29942/807091879.py", line 21, in <module>
    df = pl.read_parquet(temp_path)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/_utils/deprecation.py", line 128, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/_utils/deprecation.py", line 128, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/io/parquet/functions.py", line 293, in read_parquet
    return lf.collect()
           ^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/_utils/deprecation.py", line 97, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/sit

In [16]:
import sys
import os

# 1. 현재 주피터 커널 환경에 pyarrow 강제 설치
print(f"🔧 현재 커널({sys.executable})에 pyarrow 설치 시도...")
!{sys.executable} -m pip install --upgrade pyarrow

# 2. 설치 후 경로 재인식 (중요)
import site
import importlib
importlib.reload(site)

try:
    import pyarrow as pa
    print(f"✅ PyArrow 로드 성공 (버전: {pa.__version__})")
except ImportError:
    print("❌ 여전히 로드 실패. 수동으로 커널을 Restart 해야 할 수도 있습니다.")

# ==========================================
# 3. 데이터 연산 재개 (DuckDB)
# ==========================================
import polars as pl
import duckdb
import gc

base_dir = "/home/tracerofjageum/"
temp_path = base_dir + "temp_stacked_full.parquet"
output_path = base_dir + "output_features_full.parquet"

print("--- Phase 2: DuckDB SQL 기반 최종 집계 ---")

try:
    df = pl.read_parquet(temp_path)
    
    # 타입 정규화
    df = df.with_columns([
        pl.col("amount").cast(pl.Float64),
        pl.col("amt_out_pre").cast(pl.Float64),
        pl.col("amt_in_pre").cast(pl.Float64),
        pl.col("cnt_out_pre").cast(pl.Int64),
        pl.col("cnt_in_pre").cast(pl.Int64),
    ])

    # 1차 집계
    df_agg = df.group_by(["account_id", "time_group"]).agg([
        pl.len().cast(pl.Int64).alias("cnt_1h"),
        pl.col("amount").sum().alias("sum_1h"),
        pl.col("amount").max().alias("max_1h"),
        pl.col("amount").mean().alias("avg_1h"),
        pl.col("cnt_out_pre").sum().alias("cnt_out_1h"),
        pl.col("cnt_in_pre").sum().alias("cnt_in_1h"),
        pl.col("amt_out_pre").sum().alias("sum_out_1h"),
        pl.col("amt_in_pre").sum().alias("sum_in_1h"),
        pl.col("is_night").sum().alias("cnt_night"),
        pl.col("is_weekend").sum().alias("cnt_weekend"),
        pl.col("is_small_tx").sum().alias("cnt_small_tx"),
        pl.col("is_risk_country").sum().alias("cnt_risk_country"),
        pl.col("Is_High_Risk_Format").sum().alias("cnt_risk_format"),
        pl.col("log_amount").mean().alias("avg_log_amount"),
        pl.col("Entity_Account_Count").max().alias("entity_acct_cnt"),
        pl.col("Bank_Type_Code").max().alias("bank_type"),
        pl.col("Is Laundering").max().alias("is_laundering")
    ]).sort(["account_id", "time_group"])

    del df
    gc.collect()

    # 
    # 2. DuckDB Rolling (PyArrow를 징검다리로 사용)
    query = """
    SELECT 
        *,
        SUM(sum_1h) OVER (PARTITION BY account_id ORDER BY time_group 
            RANGE BETWEEN INTERVAL '24 HOURS' PRECEDING AND INTERVAL '1 HOUR' PRECEDING) as sum_24h,
        SUM(cnt_1h) OVER (PARTITION BY account_id ORDER BY time_group 
            RANGE BETWEEN INTERVAL '24 HOURS' PRECEDING AND INTERVAL '1 HOUR' PRECEDING) as cnt_24h,
        AVG(sum_1h) OVER (PARTITION BY account_id ORDER BY time_group 
            RANGE BETWEEN INTERVAL '7 DAYS' PRECEDING AND INTERVAL '1 HOUR' PRECEDING) as avg_7d
    FROM df_agg
    """
    
    # 여기서 PyArrow가 필요함
    df_final = duckdb.sql(query).pl()

    # 3. 최종 비율 계산 및 저장
    df_final = df_final.with_columns([
        (pl.col("cnt_1h") / (pl.col("cnt_24h") / 24 + 1)).alias("burst_ratio"),
        (pl.col("sum_in_1h") / (pl.col("sum_out_1h") + 1)).alias("in_out_ratio"),
        (pl.col("cnt_small_tx") / (pl.col("cnt_1h") + 1)).alias("ratio_small_tx"),
        (pl.col("cnt_risk_country") / (pl.col("cnt_1h") + 1)).alias("ratio_risk_country"),
        (pl.col("sum_1h") / (pl.col("avg_7d") + 1.0)).alias("amt_to_avg_ratio")
    ])

    if os.path.exists(output_path): os.remove(output_path)
    df_final.write_parquet(output_path)
    
    print(f"✅ 전처리 끝! 최종 Shape: {df_final.shape}")
    if os.path.exists(temp_path): os.remove(temp_path)

except Exception as e:
    print(f"❌ 에러: {e}")

🔧 현재 커널(/home/tracerofjageum/my-lab/.venv/bin/python3)에 pyarrow 설치 시도...
✅ PyArrow 로드 성공 (버전: 23.0.0)
--- Phase 2: DuckDB SQL 기반 최종 집계 ---
❌ 에러: No such file or directory (os error 2): /home/tracerofjageum/temp_stacked_full.parquet


In [17]:
import polars as pl
import os
import gc

base_dir = "/home/tracerofjageum/"
temp_path = base_dir + "temp_stacked_full.parquet"
output_path = base_dir + "output_features_full.parquet"

print("--- Phase 2: Polars Native 윈도우 연산 (버그 우회 버전) ---")

try:
    # 1. 데이터 로드
    df = pl.read_parquet(temp_path)
    
    # 2. 1차 집계 (이건 에러 안 남)
    df_agg = df.group_by(["account_id", "time_group"]).agg([
        pl.len().alias("cnt_1h"),
        pl.col("amount").sum().alias("sum_1h"),
        pl.col("amount").max().alias("max_1h"),
        pl.col("amount").mean().alias("avg_1h"),
        pl.col("cnt_out_pre").sum().alias("cnt_out_1h"),
        pl.col("cnt_in_pre").sum().alias("cnt_in_1h"),
        pl.col("amt_out_pre").sum().alias("sum_out_1h"),
        pl.col("amt_in_pre").sum().alias("sum_in_1h"),
        pl.col("is_night").sum().alias("cnt_night"),
        pl.col("is_weekend").sum().alias("cnt_weekend"),
        pl.col("is_small_tx").sum().alias("cnt_small_tx"),
        pl.col("is_risk_country").sum().alias("cnt_risk_country"),
        pl.col("Is_High_Risk_Format").sum().alias("cnt_risk_format"),
        pl.col("log_amount").mean().alias("avg_log_amount"),
        pl.col("Entity_Account_Count").max().alias("entity_acct_cnt"),
        pl.col("Bank_Type_Code").max().alias("bank_type"),
        pl.col("Is Laundering").max().alias("is_laundering")
    ]).sort(["account_id", "time_group"])

    del df
    gc.collect()

    # 3. [핵심] Rolling 연산 (에러 원천 차단)
    # rolling().sum() 대신 rolling().map_batches()를 사용하여 
    # Polars의 내부 최적화 버그를 우회합니다.
    print("🌊 Rolling Window 계산 중 (버그 우회 로직)...")

    def safe_rolling_sum(s):
        # Polars의 C++ 엔진 대신 넘파이/파이썬 레벨에서 합계를 구함
        return s.to_numpy().sum()

    # 24시간 합계, 7일 평균 등을 계산
    # 이 방식은 속도는 조금 느릴 수 있지만, 'list[f64]' 에러는 절대로 나지 않습니다.
    df_final = df_agg.with_columns([
        pl.col("sum_1h").rolling(index_column="time_group", period="24h", closed="left")
        .map_batches(lambda s: s.sum()) # sum() 대신 map_batches 사용
        .over("account_id").fill_null(0.0).alias("sum_24h"),
        
        pl.col("cnt_1h").rolling(index_column="time_group", period="24h", closed="left")
        .map_batches(lambda s: s.sum())
        .over("account_id").fill_null(0).alias("cnt_24h"),
        
        pl.col("sum_1h").rolling(index_column="time_group", period="7d", closed="left")
        .map_batches(lambda s: s.mean())
        .over("account_id").fill_null(0.0).alias("avg_7d")
    ])

    # 4. 최종 비율 계산
    print("➗ 최종 파생 변수 계산 중...")
    df_final = df_final.with_columns([
        (pl.col("cnt_1h") / (pl.col("cnt_24h") / 24 + 1)).alias("burst_ratio"),
        (pl.col("sum_in_1h") / (pl.col("sum_out_1h") + 1)).alias("in_out_ratio"),
        (pl.col("cnt_small_tx") / (pl.col("cnt_1h") + 1)).alias("ratio_small_tx"),
        (pl.col("cnt_risk_country") / (pl.col("cnt_1h") + 1)).alias("ratio_risk_country"),
        (pl.col("sum_1h") / (pl.col("avg_7d") + 1.0)).alias("amt_to_avg_ratio")
    ])

    # 5. 저장
    if os.path.exists(output_path): os.remove(output_path)
    df_final.write_parquet(output_path)
    
    print(f"✅ 드디어 성공! 최종 Shape: {df_final.shape}")
    if os.path.exists(temp_path): os.remove(temp_path)

except Exception as e:
    print(f"❌ 또 에러 발생...: {e}")
    # 만약 여기서도 에러가 나면, 최후의 수단으로 데이터 크기를 줄여서 시도해야 할 수도 있습니다.

--- Phase 2: Polars Native 윈도우 연산 (버그 우회 버전) ---
❌ 또 에러 발생...: No such file or directory (os error 2): /home/tracerofjageum/temp_stacked_full.parquet


In [18]:
import polars as pl
import os
import gc

base_dir = "/home/tracerofjageum/"
temp_path = base_dir + "temp_stacked_full.parquet"
output_path = base_dir + "output_features_full.parquet"

print("--- Phase 2: Polars Native (타입 명시 버전) ---")

try:
    # 1. 데이터 로드
    df_agg = pl.read_parquet(temp_path).group_by(["account_id", "time_group"]).agg([
        pl.len().alias("cnt_1h"),
        pl.col("amount").sum().alias("sum_1h"),
        pl.col("amount").max().alias("max_1h"),
        pl.col("amount").mean().alias("avg_1h"),
        pl.col("cnt_out_pre").sum().alias("cnt_out_1h"),
        pl.col("cnt_in_pre").sum().alias("cnt_in_1h"),
        pl.col("amt_out_pre").sum().alias("sum_out_1h"),
        pl.col("amt_in_pre").sum().alias("sum_in_1h"),
        pl.col("is_night").sum().alias("cnt_night"),
        pl.col("is_weekend").sum().alias("cnt_weekend"),
        pl.col("is_small_tx").sum().alias("cnt_small_tx"),
        pl.col("is_risk_country").sum().alias("cnt_risk_country"),
        pl.col("Is_High_Risk_Format").sum().alias("cnt_risk_format"),
        pl.col("log_amount").mean().alias("avg_log_amount"),
        pl.col("Entity_Account_Count").max().alias("entity_acct_cnt"),
        pl.col("Bank_Type_Code").max().alias("bank_type"),
        pl.col("Is Laundering").max().alias("is_laundering")
    ]).sort(["account_id", "time_group"])

    gc.collect()

    print("🌊 Rolling Window 계산 중 (타입 명시적 지정)...")

    # 
    # sum()과 mean()의 결과 타입을 명확히 알려줌으로써 'list[f32]' 에러와 'infer type' 에러를 동시에 잡습니다.
    df_final = df_agg.with_columns([
        pl.col("sum_1h").rolling(index_column="time_group", period="24h", closed="left")
        .map_batches(lambda s: s.sum(), return_dtype=pl.Float64) # Float64로 명시
        .over("account_id").fill_null(0.0).alias("sum_24h"),
        
        pl.col("cnt_1h").rolling(index_column="time_group", period="24h", closed="left")
        .map_batches(lambda s: s.sum(), return_dtype=pl.Int64) # Int64로 명시
        .over("account_id").fill_null(0).alias("cnt_24h"),
        
        pl.col("sum_1h").rolling(index_column="time_group", period="7d", closed="left")
        .map_batches(lambda s: s.mean(), return_dtype=pl.Float64)
        .over("account_id").fill_null(0.0).alias("avg_7d")
    ])

    print("➗ 최종 파생 변수 계산 중...")
    df_final = df_final.with_columns([
        (pl.col("cnt_1h") / (pl.col("cnt_24h") / 24 + 1)).alias("burst_ratio"),
        (pl.col("sum_in_1h") / (pl.col("sum_out_1h") + 1)).alias("in_out_ratio"),
        (pl.col("cnt_small_tx") / (pl.col("cnt_1h") + 1)).alias("ratio_small_tx"),
        (pl.col("cnt_risk_country") / (pl.col("cnt_1h") + 1)).alias("ratio_risk_country"),
        (pl.col("sum_1h") / (pl.col("avg_7d") + 1.0)).alias("amt_to_avg_ratio")
    ])

    if os.path.exists(output_path): os.remove(output_path)
    df_final.write_parquet(output_path)
    
    print(f"✅ 드디어!! 성공했습니다! 최종 Shape: {df_final.shape}")
    
    # 임시 파일 삭제
    if os.path.exists(temp_path): os.remove(temp_path)

except Exception as e:
    print(f"❌ 또 에러 발생...: {e}")
    import traceback
    traceback.print_exc()

--- Phase 2: Polars Native (타입 명시 버전) ---
❌ 또 에러 발생...: No such file or directory (os error 2): /home/tracerofjageum/temp_stacked_full.parquet


Traceback (most recent call last):
  File "/tmp/ipykernel_29942/3863929719.py", line 13, in <module>
    df_agg = pl.read_parquet(temp_path).group_by(["account_id", "time_group"]).agg([
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/_utils/deprecation.py", line 128, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/_utils/deprecation.py", line 128, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/io/parquet/functions.py", line 293, in read_parquet
    return lf.collect()
           ^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/_utils/deprecation.py", line 97, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File

In [19]:
import polars as pl
import os
import gc

base_dir = "/home/tracerofjageum/"
temp_path = base_dir + "temp_stacked_full.parquet"
output_path = base_dir + "output_features_full.parquet"

print("--- Phase 2: 집계 및 누적합 기반 윈도우 연산 (안전 모드) ---")

try:
    # 1. 집계 데이터 로드 및 1차 요약
    # 여기서 에러가 났던 컬럼명들을 원본 컬럼명(is_...)으로 수정했습니다.
    df_agg = pl.read_parquet(temp_path).group_by(["account_id", "time_group"]).agg([
        pl.len().alias("cnt_1h"),
        pl.col("amount").sum().alias("sum_1h"),
        pl.col("amt_out_pre").sum().alias("sum_out_1h"),
        pl.col("amt_in_pre").sum().alias("sum_in_1h"),
        pl.col("is_small_tx").sum().alias("cnt_small_tx"),     # 원본: is_small_tx
        pl.col("is_risk_country").sum().alias("cnt_risk_country"), # 원본: is_risk_country
        pl.col("is_night").sum().alias("cnt_night"),           # 원본: is_night
        pl.col("amount").max().alias("max_1h"),
        pl.col("amount").mean().alias("avg_1h"),
        pl.col("log_amount").mean().alias("avg_log_amount"),
        pl.col("Entity_Account_Count").max().alias("entity_acct_cnt"),
        pl.col("Bank_Type_Code").max().alias("bank_type"),
        pl.col("Is Laundering").max().alias("is_laundering")
    ]).sort(["account_id", "time_group"])

    print("🔄 누적합(Cumulative Sum) 기반 윈도우 연산 중...")
    
    # [Image of Slapping Window using Cumulative Sum]
    # 2. Rolling().sum() 대신 Cumulative Sum - Cumulative Sum(Lagged) 방식으로 구간 합 계산
    # 이 방식은 리스트 에러를 유발하는 rolling 엔진의 버그를 원천 차단합니다.
    
    # 24시간 전 시점의 누적값을 가져오기 위해 오프셋 계산
    # (여기서는 각 계좌별로 시간 순서대로 정렬되어 있으므로 윈도우 연산이 안정적입니다.)
    
    df_final = df_agg.with_columns([
        # 24시간 합계 (현재 행 제외 과거 24행 - 시간 단위 그룹화이므로 24행이 24시간임)
        pl.col("sum_1h").cum_sum().over("account_id").alias("_tmp_cumsum"),
        pl.col("cnt_1h").cum_sum().over("account_id").alias("_tmp_cntcumsum")
    ]).with_columns([
        # (현재 누적합 - 24시간 전 누적합) = 최근 24시간의 합
        (pl.col("_tmp_cumsum").shift(1) - pl.col("_tmp_cumsum").shift(25)).over("account_id").fill_null(0.0).alias("sum_24h"),
        (pl.col("_tmp_cntcumsum").shift(1) - pl.col("_tmp_cntcumsum").shift(25)).over("account_id").fill_null(0).alias("cnt_24h"),
        # 7일 평균 (168시간)
        (pl.col("_tmp_cumsum").shift(1) - pl.col("_tmp_cumsum").shift(169)).over("account_id").fill_null(0.0).alias("_sum_7d")
    ]).with_columns([
        (pl.col("_sum_7d") / 168).alias("avg_7d")
    ]).drop(["_tmp_cumsum", "_tmp_cntcumsum", "_sum_7d"])

    # 3. 최종 비율 계산
    print("➗ 최종 파생 변수 계산 및 저장 중...")
    df_final = df_final.with_columns([
        (pl.col("cnt_1h") / (pl.col("cnt_24h").cast(pl.Float64) / 24 + 1)).alias("burst_ratio"),
        (pl.col("sum_in_1h") / (pl.col("sum_out_1h") + 1)).alias("in_out_ratio"),
        (pl.col("cnt_small_tx") / (pl.col("cnt_1h") + 1)).alias("ratio_small_tx"),
        (pl.col("cnt_risk_country") / (pl.col("cnt_1h") + 1)).alias("ratio_risk_country"),
        (pl.col("sum_1h") / (pl.col("avg_7d") + 1.0)).alias("amt_to_avg_ratio")
    ])

    if os.path.exists(output_path): os.remove(output_path)
    df_final.write_parquet(output_path)
    
    print(f"✅ [대성공] 전처리가 완료되었습니다! 최종 Shape: {df_final.shape}")
    if os.path.exists(temp_path): os.remove(temp_path)

except Exception as e:
    print(f"❌ 또 에러 발생...: {e}")
    import traceback
    traceback.print_exc()

--- Phase 2: 집계 및 누적합 기반 윈도우 연산 (안전 모드) ---
❌ 또 에러 발생...: No such file or directory (os error 2): /home/tracerofjageum/temp_stacked_full.parquet


Traceback (most recent call last):
  File "/tmp/ipykernel_29942/3299966051.py", line 14, in <module>
    df_agg = pl.read_parquet(temp_path).group_by(["account_id", "time_group"]).agg([
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/_utils/deprecation.py", line 128, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/_utils/deprecation.py", line 128, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/io/parquet/functions.py", line 293, in read_parquet
    return lf.collect()
           ^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/_utils/deprecation.py", line 97, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File

In [20]:
import polars as pl

# 1. 데이터 로드
output_path = "/home/tracerofjageum/output_features_full.parquet"
df = pl.read_parquet(output_path)

# 2. 기본 정보 출력
print(f"✅ 데이터 Shape: {df.shape}")
print(f"✅ 전체 컬럼 수: {len(df.columns)}개")

# 3. 요청하신 핵심 피처 존재 여부 전수 조사
check_list = [
    # 시간/활동성
    "cnt_1h", "cnt_night", "cnt_weekend", "burst_ratio", 
    # 금액/통계
    "avg_log_amount", "sum_1h", "std_1h", "in_out_ratio", "cnt_small_tx", "ratio_small_tx",
    # 네트워크/위험
    "degree_1h", "cnt_risk_country", "ratio_risk_country", "cnt_risk_format",
    # 행동 이탈
    "amt_to_avg_ratio", "avg_7d", "entity_acct_cnt"
]

print("\n--- [주요 피처 생성 여부 확인] ---")
found = [c for c in check_list if c in df.columns]
missing = [c for c in check_list if c not in df.columns]
print(f"🟢 생성 성공 ({len(found)}개): {found}")
if missing:
    print(f"🔴 누락/명칭변경 ({len(missing)}개): {missing}")

# 4. 데이터 샘플 및 통계 요약 (상위 5개 행)
print("\n--- [데이터 샘플 미리보기] ---")
# 주요 피처 몇 가지만 골라서 샘플 확인
display_cols = ["account_id", "time_group", "sum_1h", "burst_ratio", "in_out_ratio", "ratio_risk_country", "is_laundering"]
print(df.select([c for c in display_cols if c in df.columns]).head(5))

# 5. 수치 데이터 분포 확인 (이상치/결측치 체크)
print("\n--- [주요 피처 통계 요약] ---")
print(df.select(found[:8]).describe()) # 상위 8개 피처에 대한 통계

# 6. 타겟 데이터(자금세탁) 비중 확인
if "is_laundering" in df.columns:
    laundering_counts = df["is_laundering"].value_counts()
    print("\n--- [타겟(자금세탁) 라벨 분포] ---")
    print(laundering_counts)
    

✅ 데이터 Shape: (28756304, 33)
✅ 전체 컬럼 수: 33개

--- [주요 피처 생성 여부 확인] ---
🟢 생성 성공 (17개): ['cnt_1h', 'cnt_night', 'cnt_weekend', 'burst_ratio', 'avg_log_amount', 'sum_1h', 'std_1h', 'in_out_ratio', 'cnt_small_tx', 'ratio_small_tx', 'degree_1h', 'cnt_risk_country', 'ratio_risk_country', 'cnt_risk_format', 'amt_to_avg_ratio', 'avg_7d', 'entity_acct_cnt']

--- [데이터 샘플 미리보기] ---
shape: (5, 7)
┌────────────┬──────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ account_id ┆ time_group   ┆ sum_1h       ┆ burst_ratio ┆ in_out_rati ┆ ratio_risk_ ┆ is_launderi │
│ ---        ┆ ---          ┆ ---          ┆ ---         ┆ o           ┆ country     ┆ ng          │
│ str        ┆ datetime[μs] ┆ f64          ┆ decimal[38, ┆ ---         ┆ ---         ┆ ---         │
│            ┆              ┆              ┆ 0]          ┆ f64         ┆ f64         ┆ u8          │
╞════════════╪══════════════╪══════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ 806CE8

In [21]:
import polars as pl

# 데이터 로드
df = pl.read_parquet("/home/tracerofjageum/output_features_full.parquet")

# 위험 국가 거래 비중이 0보다 큰(즉, 매핑이 성공한) 데이터만 필터링
risk_check = df.filter(pl.col("ratio_risk_country") > 0).select([
    "account_id", "time_group", "cnt_risk_country", "ratio_risk_country"
]).head(10)

print("--- [국가 매핑 결과 확인] ---")
if len(risk_check) > 0:
    print(risk_check)
else:
    print("위험 국가로 매핑된 거래가 이 샘플에는 없습니다. (전체 데이터를 확인해야 할 수 있음)")

--- [국가 매핑 결과 확인] ---
shape: (10, 4)
┌────────────┬─────────────────────┬──────────────────┬────────────────────┐
│ account_id ┆ time_group          ┆ cnt_risk_country ┆ ratio_risk_country │
│ ---        ┆ ---                 ┆ ---              ┆ ---                │
│ str        ┆ datetime[μs]        ┆ i64              ┆ f64                │
╞════════════╪═════════════════════╪══════════════════╪════════════════════╡
│ 807118BB0  ┆ 2022-09-01 03:00:00 ┆ 4                ┆ 0.8                │
│ 807118BB0  ┆ 2022-09-01 23:00:00 ┆ 1                ┆ 0.5                │
│ 807118BB0  ┆ 2022-09-02 02:00:00 ┆ 4                ┆ 0.8                │
│ 807118BB0  ┆ 2022-09-02 10:00:00 ┆ 1                ┆ 0.5                │
│ 807118BB0  ┆ 2022-09-02 17:00:00 ┆ 2                ┆ 0.666667           │
│ 807118BB0  ┆ 2022-09-05 00:00:00 ┆ 4                ┆ 0.8                │
│ 807118BB0  ┆ 2022-09-05 17:00:00 ┆ 1                ┆ 0.5                │
│ 807118BB0  ┆ 2022-09-06 02:00:00 ┆ 1 

In [22]:
import polars as pl

# 데이터 로드
df = pl.read_parquet("/home/tracerofjageum/output_features_full.parquet")

# 1. 실제로 존재하는 핵심 '위험' 지표들만 추출
# ratio_risk_country: 위험 국가 거래 비중
# in_out_ratio: 입금 대비 출금 비율 (1.0에 가까우면 통로 의심)
# burst_ratio: 평소 대비 거래 폭증 정도
check_suspicious = df.filter(pl.col("ratio_risk_country") > 0).select([
    "account_id", "time_group", "cnt_1h", "ratio_risk_country", "in_out_ratio", "burst_ratio", "is_laundering"
]).sort("ratio_risk_country", descending=True).head(10)

print("--- [생성 완료된 핵심 위험 지표 확인] ---")
if len(check_suspicious) > 0:
    print(check_suspicious)
else:
    print("샘플 내 위험 지표 데이터가 없습니다. 필터 조건을 완화하여 확인해보세요.")

--- [생성 완료된 핵심 위험 지표 확인] ---
shape: (10, 7)
┌────────────┬───────────────┬────────┬───────────────┬──────────────┬──────────────┬──────────────┐
│ account_id ┆ time_group    ┆ cnt_1h ┆ ratio_risk_co ┆ in_out_ratio ┆ burst_ratio  ┆ is_launderin │
│ ---        ┆ ---           ┆ ---    ┆ untry         ┆ ---          ┆ ---          ┆ g            │
│ str        ┆ datetime[μs]  ┆ i64    ┆ ---           ┆ f64          ┆ decimal[38,0 ┆ ---          │
│            ┆               ┆        ┆ f64           ┆              ┆ ]            ┆ u8           │
╞════════════╪═══════════════╪════════╪═══════════════╪══════════════╪══════════════╪══════════════╡
│ 1004287C8  ┆ 2022-09-10    ┆ 108    ┆ 0.990826      ┆ 0.0          ┆ 0            ┆ 0            │
│            ┆ 00:00:00      ┆        ┆               ┆              ┆              ┆              │
│ 1004287C8  ┆ 2022-09-11    ┆ 104    ┆ 0.990476      ┆ 0.0          ┆ 1            ┆ 1            │
│            ┆ 08:00:00      ┆        ┆        

In [23]:
import polars as pl
import os
import gc
import numpy as np

# 경로 설정
base_dir = "/home/tracerofjageum/"
source_path = base_dir + "HI-Medium_Master.parquet"
output_path = base_dir + "aml_features_final_advanced.parquet"

print("--- [최종 공정] 실제 컬럼명 반영 및 고도화 피처 생성 시작 ---")

try:
    # 1. 원본 데이터 로드
    df = pl.read_parquet(source_path)
    
    # 시간 표준화 및 정렬
    if df["Timestamp"].dtype == pl.Utf8:
        df = df.with_columns(pl.col("Timestamp").str.to_datetime())
    df = df.sort("Timestamp")

    # -------------------------------------------------------
    # 2. 4단계: 관점 변환 (Stacking - 송금/수금 2줄로 복사)
    # -------------------------------------------------------
    print("🔄 데이터 관점 변환 (Edge -> Node) 중...")
    
    # 송금자(Out) 관점
    df_out = df.select([
        pl.all(),
        pl.col("from_acc").alias("account_id"),
        pl.col("to_acc").alias("counterparty"),
        pl.lit(1).cast(pl.UInt8).alias("_is_out"),
        pl.col("Amount Paid").alias("_amt")
    ])
    
    # 수금자(In) 관점
    df_in = df.select([
        pl.all(),
        pl.col("to_acc").alias("account_id"),
        pl.col("from_acc").alias("counterparty"),
        pl.lit(0).cast(pl.UInt8).alias("_is_out"),
        pl.col("Amount Received").alias("_amt")
    ])
    
    df_stacked = pl.concat([df_out, df_in])
    del df, df_out, df_in
    gc.collect()

    # -------------------------------------------------------
    # 3. 고도화 사전 작업 (시간차, 벤포드, 요일)
    # -------------------------------------------------------
    print("🔧 고도화 피처 사전 연산 중 (Time Delta, Benford)...")
    df_stacked = df_stacked.with_columns([
        pl.col("Timestamp").dt.truncate("1h").alias("time_group"),
        pl.col("Timestamp").dt.weekday().alias("weekday"),
        # 벤포드 법칙: 금액 첫 자리 (문자열 변환 후 추출)
        pl.col("_amt").cast(pl.Utf8).str.slice(0, 1).alias("_first_digit"),
        # 계좌별 직전 거래와의 시간 간격 (초 단위)
        (pl.col("Timestamp").diff().dt.total_seconds()).over("account_id").fill_null(0).alias("_time_delta")
    ])

    # -------------------------------------------------------
    # 4. 1시간 단위 정밀 집계 (Upgrade 피처 로직 포함)
    # -------------------------------------------------------
    print("🔄 1시간 단위 고도화 집계 수행 중...")
    
    # 쪼개기(Structuring) 기준: $10,000
    SMALL_TX_LIMIT = 10000 

    df_agg = df_stacked.group_by(["account_id", "time_group"]).agg([
        # [Time & Frequency]
        pl.len().alias("cnt_1h"),
        pl.col("_time_delta").mean().alias("time_delta_mean"),
        pl.col("_time_delta").std().fill_null(0).alias("time_delta_std"),
        pl.col("_time_delta").min().alias("min_inter_tx_gap"), # Pass-through 탐지
        (pl.col("Timestamp").dt.hour() < 6).sum().alias("cnt_night"),
        (pl.col("weekday") >= 6).sum().alias("cnt_weekend"),
        
        # [Amount & Statistics]
        pl.col("_amt").sum().alias("sum_1h"),
        pl.col("_amt").max().alias("max_1h"),
        pl.col("_amt").std().fill_null(0).alias("std_1h"),
        pl.col("_amt").filter(pl.col("_is_out") == 1).sum().alias("sum_out_1h"),
        pl.col("_amt").filter(pl.col("_is_out") == 0).sum().alias("sum_in_1h"),
        
        # [Risk & Network Pattern]
        (pl.col("_amt") < SMALL_TX_LIMIT).sum().alias("cnt_small_tx"),
        pl.col("counterparty").n_unique().alias("degree_1h"),
        # Entity 정보 활용 (src_entity_id나 dst_entity_id 중 하나라도 있으면)
        pl.col("src_entity_id").n_unique().alias("entity_acct_cnt"), 
        
        # [Benford & Log]
        (pl.col("_first_digit") == "1").mean().alias("benford_1_ratio"),
        (pl.col("_amt") + 1).log().mean().alias("avg_log_amount"),
        
        # Target
        pl.col("Is Laundering").max().alias("is_laundering")
    ])

    del df_stacked
    gc.collect()

    # -------------------------------------------------------
    # 5. 장기 윈도우 & 최종 비율 계산 (Burst, Ratio 등)
    # -------------------------------------------------------
    print("🌊 장기 패턴 및 최종 업그레이드 비율 산출 중...")
    
    df_final = df_agg.sort(["account_id", "time_group"]).with_columns([
        pl.col("sum_1h").cum_sum().over("account_id").alias("_sum_cum"),
        pl.col("cnt_1h").cum_sum().over("account_id").alias("_cnt_cum")
    ]).with_columns([
        (pl.col("_sum_cum").shift(1) - pl.col("_sum_cum").shift(25)).over("account_id").fill_null(0).alias("sum_24h"),
        (pl.col("_cnt_cum").shift(1) - pl.col("_cnt_cum").shift(25)).over("account_id").fill_null(0).alias("cnt_24h"),
        (pl.col("_sum_cum").shift(1) - pl.col("_sum_cum").shift(169)).over("account_id").fill_null(0).alias("_sum_7d")
    ]).with_columns([
        (pl.col("_sum_7d") / 168).alias("avg_7d"),
        (pl.col("cnt_1h") / (pl.col("cnt_24h") / 24 + 1)).alias("burst_ratio"),
        (pl.col("sum_in_1h") / (pl.col("sum_out_1h") + 1)).alias("in_out_balance_ratio"),
        (pl.col("time_delta_std") / (pl.col("time_delta_mean") + 1e-6)).alias("inter_arrival_cv"),
        (pl.col("benford_1_ratio") - 0.301).abs().alias("benford_deviation")
    ])

    # 임시 컬럼 제거 후 저장
    final_cols = [c for c in df_final.columns if not c.startswith("_")]
    df_final.select(final_cols).write_parquet(output_path)
    
    print("-" * 30)
    print(f"✅ [미반영 복구 완료] 최종 파일: {output_path}")
    print(f"📊 최종 피처 수: {len(df_final.select(final_cols).columns)}개")
    print("-" * 30)

except Exception as e:
    print(f"❌ 에러 발생: {e}")
    import traceback
    traceback.print_exc()

--- [최종 공정] 실제 컬럼명 반영 및 고도화 피처 생성 시작 ---
🔄 데이터 관점 변환 (Edge -> Node) 중...
❌ 에러 발생: type Float64 is incompatible with expected type Float32


Traceback (most recent call last):
  File "/tmp/ipykernel_29942/1025300551.py", line 45, in <module>
    df_stacked = pl.concat([df_out, df_in])
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/functions/eager.py", line 234, in concat
    out = wrap_df(plr.concat_df(elems))
                  ^^^^^^^^^^^^^^^^^^^^
polars.exceptions.SchemaError: type Float64 is incompatible with expected type Float32


In [24]:
import polars as pl
import os
import gc

# 경로 설정
source_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
output_path = "/home/tracerofjageum/aml_features_final_advanced.parquet"

print("--- [최종 공정] 모든 업그레이드 피처 + 국가 매핑 + 타입 통합 시작 ---")

try:
    # 1. 데이터 로드
    df = pl.read_parquet(source_path)
    
    # 타입 및 시간 표준화 (금액은 미리 Float64로 통일)
    df = df.with_columns([
        pl.col("Timestamp").str.to_datetime(),
        pl.col("Amount Paid").cast(pl.Float64),
        pl.col("Amount Received").cast(pl.Float64)
    ]).sort("Timestamp")

    # -------------------------------------------------------
    # 2. 고위험 국가 매핑 (규빈님 팀 로직 반영)
    # -------------------------------------------------------
    # 은행명에 아래 키워드가 있으면 고위험 국가 거래로 간주
    risk_keywords = ["Russia", "Russian", "Mexico", "Saudi", "China", "Crypto", "Digital", "Iran", "Korea"]
    
    df = df.with_columns([
        pl.col("From Bank").str.contains("(?i)" + "|".join(risk_keywords)).alias("_from_risk"),
        pl.col("To Bank").str.contains("(?i)" + "|".join(risk_keywords)).alias("_to_risk"),
        (pl.col("From Bank") != pl.col("To Bank")).alias("_is_cross_border")
    ])

    # -------------------------------------------------------
    # 3. 관점 변환 (Stacking: Edge -> Node)
    # -------------------------------------------------------
    print("🔄 데이터 관점 변환 및 Stacking 중...")
    
    # 송금자 관점
    df_out = df.select([
        pl.all(),
        pl.col("from_acc").alias("account_id"),
        pl.col("to_acc").alias("counterparty"),
        pl.lit(1).cast(pl.UInt8).alias("_is_out"),
        pl.col("Amount Paid").alias("_amt"),
        pl.col("_from_risk").alias("_is_risk")
    ])
    
    # 수금자 관점
    df_in = df.select([
        pl.all(),
        pl.col("to_acc").alias("account_id"),
        pl.col("from_acc").alias("counterparty"),
        pl.lit(0).cast(pl.UInt8).alias("_is_out"),
        pl.col("Amount Received").alias("_amt"),
        pl.col("_to_risk").alias("_is_risk")
    ])
    
    df_stacked = pl.concat([df_out, df_in])
    del df, df_out, df_in
    gc.collect()

    # -------------------------------------------------------
    # 4. 사전 연산 (요일, 벤포드, 시간차)
    # -------------------------------------------------------
    print("🔧 초정밀 피처 사전 연산 중...")
    df_stacked = df_stacked.with_columns([
        pl.col("Timestamp").dt.truncate("1h").alias("time_group"),
        pl.col("Timestamp").dt.weekday().alias("weekday"),
        pl.col("_amt").cast(pl.Utf8).str.slice(0, 1).alias("_first_digit"),
        (pl.col("Timestamp").diff().dt.total_seconds()).over("account_id").fill_null(0).alias("_time_delta")
    ])

    # -------------------------------------------------------
    # 5. 1시간 단위 집계 (표에 있는 모든 상세 로직)
    # -------------------------------------------------------
    print("🔄 고도화 집계 수행 중 (Structuring, Pass-through, Layering)...")
    
    df_agg = df_stacked.group_by(["account_id", "time_group"]).agg([
        # [활동성 및 시간 간격]
        pl.len().alias("cnt_1h"),
        pl.col("_time_delta").mean().alias("time_delta_mean"),
        pl.col("_time_delta").std().fill_null(0).alias("time_delta_std"),
        pl.col("_time_delta").min().alias("min_inter_tx_gap"),
        (pl.col("Timestamp").dt.hour() < 6).sum().alias("cnt_night"),
        (pl.col("weekday") >= 6).sum().alias("cnt_weekend"),
        
        # [금액 통계]
        pl.col("_amt").sum().alias("sum_1h"),
        pl.col("_amt").max().alias("max_1h"),
        pl.col("_amt").std().fill_null(0).alias("std_1h"),
        pl.col("_amt").filter(pl.col("_is_out") == 1).sum().alias("sum_out_1h"),
        pl.col("_amt").filter(pl.col("_is_out") == 0).sum().alias("sum_in_1h"),
        
        # [위험 요소 및 네트워크]
        (pl.col("_amt") < 10000).sum().alias("cnt_small_tx"),
        pl.col("_is_risk").sum().alias("cnt_risk_country"),
        pl.col("_is_cross_border").mean().alias("ratio_cross_border"),
        pl.col("counterparty").n_unique().alias("degree_1h"),
        pl.col("src_entity_id").n_unique().alias("entity_acct_cnt"),
        pl.col("Is_High_Risk_Format").sum().alias("cnt_risk_format"),
        
        # [Benford & Log]
        (pl.col("_first_digit") == "1").mean().alias("benford_1_ratio"),
        (pl.col("_amt") + 1).log().mean().alias("avg_log_amount"),
        
        # Target
        pl.col("Is Laundering").max().alias("is_laundering")
    ])

    # -------------------------------------------------------
    # 6. 장기 윈도우 & 최종 비율 (Burst, CV, Deviation 등)
    # -------------------------------------------------------
    print("🌊 장기 패턴 및 최종 업그레이드 지표 산출 중...")
    
    df_final = df_agg.sort(["account_id", "time_group"]).with_columns([
        pl.col("sum_1h").cum_sum().over("account_id").alias("_sum_cum"),
        pl.col("cnt_1h").cum_sum().over("account_id").alias("_cnt_cum")
    ]).with_columns([
        (pl.col("_sum_cum").shift(1) - pl.col("_sum_cum").shift(25)).over("account_id").fill_null(0).alias("sum_24h"),
        (pl.col("_cnt_cum").shift(1) - pl.col("_cnt_cum").shift(25)).over("account_id").fill_null(0).alias("cnt_24h"),
        (pl.col("_sum_cum").shift(1) - pl.col("_sum_cum").shift(169)).over("account_id").fill_null(0).alias("_sum_7d")
    ]).with_columns([
        (pl.col("_sum_7d") / 168).alias("avg_7d"),
        (pl.col("cnt_1h") / (pl.col("cnt_24h") / 24 + 1)).alias("burst_ratio"),
        (pl.col("sum_in_1h") / (pl.col("sum_out_1h") + 1)).alias("in_out_balance_ratio"),
        (pl.col("time_delta_std") / (pl.col("time_delta_mean") + 1e-6)).alias("inter_arrival_cv"),
        (pl.col("benford_1_ratio") - 0.301).abs().alias("benford_deviation"),
        (pl.col("cnt_risk_country") / (pl.col("cnt_1h") + 1)).alias("ratio_risk_country")
    ])

    # 정리 및 저장
    final_cols = [c for c in df_final.columns if not c.startswith("_")]
    df_final.select(final_cols).write_parquet(output_path)
    
    print("-" * 30)
    print(f"✅ [미반영 올인원 복구] 최종 파일: {output_path}")
    print(f"📊 최종 피처 수: {len(df_final.select(final_cols).columns)}개")
    print("-" * 30)

except Exception as e:
    print(f"❌ 에러 발생: {e}")
    import traceback
    traceback.print_exc()

--- [최종 공정] 모든 업그레이드 피처 + 국가 매핑 + 타입 통합 시작 ---
❌ 에러 발생: expected String type, got: i64


Traceback (most recent call last):
  File "/tmp/ipykernel_29942/872474544.py", line 28, in <module>
    df = df.with_columns([
         ^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/dataframe/frame.py", line 10473, in with_columns
    .collect(optimizations=QueryOptFlags._eager())
     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/_utils/deprecation.py", line 97, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/lazyframe/opt_flags.py", line 326, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/lazyframe/frame.py", line 2440, in collect
    return wrap_df(ldf.collect(engine, callback))
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
pol

In [25]:
import polars as pl
import os
import gc

# 경로 설정
source_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
output_path = "/home/tracerofjageum/aml_features_final_advanced.parquet"

print("--- [최종 공정] 타입 예외 처리 및 모든 고도화 피처 통합 시작 ---")

try:
    # 1. 원본 데이터 로드
    df = pl.read_parquet(source_path)
    
    # [타입 처리] Timestamp가 숫자(i64)인 경우와 문자(String)인 경우 모두 대응
    if df["Timestamp"].dtype == pl.Int64:
        # Unix timestamp(초/밀리초)인 경우 처리 (단위에 따라 s 또는 ms 선택)
        df = df.with_columns(pl.from_epoch("Timestamp", time_unit="s"))
    else:
        df = df.with_columns(pl.col("Timestamp").str.to_datetime())

    # [타입 처리] 금액 컬럼 Float64로 통일
    df = df.with_columns([
        pl.col("Amount Paid").cast(pl.Float64),
        pl.col("Amount Received").cast(pl.Float64)
    ]).sort("Timestamp")

    # -------------------------------------------------------
    # 2. 고위험 국가 매핑 (규빈님 팀 로직 반영)
    # -------------------------------------------------------
    risk_keywords = ["Russia", "Russian", "Mexico", "Saudi", "China", "Crypto", "Digital", "Iran", "Korea"]
    
    # 은행 이름 컬럼명 확인 (From Bank, To Bank)
    df = df.with_columns([
        pl.col("From Bank").str.contains("(?i)" + "|".join(risk_keywords)).alias("_from_risk"),
        pl.col("To Bank").str.contains("(?i)" + "|".join(risk_keywords)).alias("_to_risk"),
        (pl.col("From Bank") != pl.col("To Bank")).alias("_is_cross_border")
    ])

    # -------------------------------------------------------
    # 3. 관점 변환 (Stacking: Edge -> Node)
    # -------------------------------------------------------
    print("🔄 데이터 관점 변환 및 Stacking 중...")
    
    # 공통 컬럼 리스트 (중복 방지)
    base_cols = ["Timestamp", "Payment Format", "Is Laundering", "src_entity_id", "_is_cross_border", "Is_High_Risk_Format"]
    
    df_out = df.select([
        *base_cols,
        pl.col("from_acc").alias("account_id"),
        pl.col("to_acc").alias("counterparty"),
        pl.lit(1).cast(pl.UInt8).alias("_is_out"),
        pl.col("Amount Paid").alias("_amt"),
        pl.col("_from_risk").alias("_is_risk")
    ])
    
    df_in = df.select([
        *base_cols,
        pl.col("to_acc").alias("account_id"),
        pl.col("from_acc").alias("counterparty"),
        pl.lit(0).cast(pl.UInt8).alias("_is_out"),
        pl.col("Amount Received").alias("_amt"),
        pl.col("_to_risk").alias("_is_risk")
    ])
    
    df_stacked = pl.concat([df_out, df_in])
    del df, df_out, df_in
    gc.collect()

    # -------------------------------------------------------
    # 4. 사전 연산 (요일, 벤포드, 시간차)
    # -------------------------------------------------------
    print("🔧 초정밀 피처 사전 연산 중...")
    df_stacked = df_stacked.with_columns([
        pl.col("Timestamp").dt.truncate("1h").alias("time_group"),
        pl.col("Timestamp").dt.weekday().alias("weekday"),
        # 벤포드 법칙: 금액 첫 자리 (0 제외)
        pl.col("_amt").cast(pl.Utf8).str.slice(0, 1).alias("_first_digit"),
        # 계좌별 직전 거래와의 시간 차이
        (pl.col("Timestamp").diff().dt.total_seconds()).over("account_id").fill_null(0).alias("_time_delta")
    ])

    # -------------------------------------------------------
    # 5. 1시간 단위 집계 (표의 모든 로직 100% 반영)
    # -------------------------------------------------------
    print("🔄 고도화 집계 수행 중 (Structuring, Pass-through, Layering)...")
    
    df_agg = df_stacked.group_by(["account_id", "time_group"]).agg([
        # [Time & Frequency]
        pl.len().alias("cnt_1h"),
        pl.col("_time_delta").mean().alias("time_delta_mean"),
        pl.col("_time_delta").std().fill_null(0).alias("time_delta_std"),
        pl.col("_time_delta").min().alias("min_inter_tx_gap"),
        (pl.col("Timestamp").dt.hour() < 6).sum().alias("cnt_night"),
        (pl.col("weekday") >= 6).sum().alias("cnt_weekend"),
        
        # [Amount & Statistics]
        pl.col("_amt").sum().alias("sum_1h"),
        pl.col("_amt").max().alias("max_1h"),
        pl.col("_amt").std().fill_null(0).alias("std_1h"),
        pl.col("_amt").filter(pl.col("_is_out") == 1).sum().alias("sum_out_1h"),
        pl.col("_amt").filter(pl.col("_is_out") == 0).sum().alias("sum_in_1h"),
        
        # [Risk & Network Pattern]
        (pl.col("_amt") < 10000).sum().alias("cnt_small_tx"),
        pl.col("_is_risk").sum().alias("cnt_risk_country"),
        pl.col("_is_cross_border").mean().alias("ratio_cross_border"),
        pl.col("counterparty").n_unique().alias("degree_1h"),
        pl.col("src_entity_id").n_unique().alias("entity_acct_cnt"),
        pl.col("Is_High_Risk_Format").sum().alias("cnt_risk_format"),
        
        # [Benford & Log]
        (pl.col("_first_digit") == "1").mean().alias("benford_1_ratio"),
        (pl.col("_amt") + 1).log().mean().alias("avg_log_amount"),
        
        # Target
        pl.col("Is Laundering").max().alias("is_laundering")
    ])

    # -------------------------------------------------------
    # 6. 장기 윈도우 & 최종 비율 산출
    # -------------------------------------------------------
    print("🌊 장기 패턴 및 최종 업그레이드 지표 산출 중...")
    
    df_final = df_agg.sort(["account_id", "time_group"]).with_columns([
        pl.col("sum_1h").cum_sum().over("account_id").alias("_sum_cum"),
        pl.col("cnt_1h").cum_sum().over("account_id").alias("_cnt_cum")
    ]).with_columns([
        (pl.col("_sum_cum").shift(1) - pl.col("_sum_cum").shift(25)).over("account_id").fill_null(0).alias("sum_24h"),
        (pl.col("_cnt_cum").shift(1) - pl.col("_cnt_cum").shift(25)).over("account_id").fill_null(0).alias("cnt_24h"),
        (pl.col("_sum_cum").shift(1) - pl.col("_sum_cum").shift(169)).over("account_id").fill_null(0).alias("_sum_7d")
    ]).with_columns([
        (pl.col("_sum_7d") / 168).alias("avg_7d"),
        (pl.col("cnt_1h") / (pl.col("cnt_24h") / 24 + 1)).alias("burst_ratio"),
        (pl.col("sum_in_1h") / (pl.col("sum_out_1h") + 1)).alias("in_out_balance_ratio"),
        (pl.col("time_delta_std") / (pl.col("time_delta_mean") + 1e-6)).alias("inter_arrival_cv"),
        (pl.col("benford_1_ratio") - 0.301).abs().alias("benford_deviation"),
        (pl.col("cnt_risk_country") / (pl.col("cnt_1h") + 1)).alias("ratio_risk_country")
    ])

    # 정리 및 저장
    final_cols = [c for c in df_final.columns if not c.startswith("_")]
    df_final.select(final_cols).write_parquet(output_path)
    
    print("-" * 30)
    print(f"✅ [최종 완료] 모든 피처 반영 완료: {output_path}")
    print(f"📊 최종 피처 수: {len(df_final.select(final_cols).columns)}개")
    print("-" * 30)

except Exception as e:
    print(f"❌ 에러 발생: {e}")
    import traceback
    traceback.print_exc()

--- [최종 공정] 타입 예외 처리 및 모든 고도화 피처 통합 시작 ---
❌ 에러 발생: expected String type, got: i64


Traceback (most recent call last):
  File "/tmp/ipykernel_29942/1980161328.py", line 34, in <module>
    df = df.with_columns([
         ^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/dataframe/frame.py", line 10473, in with_columns
    .collect(optimizations=QueryOptFlags._eager())
     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/_utils/deprecation.py", line 97, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/lazyframe/opt_flags.py", line 326, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/lazyframe/frame.py", line 2440, in collect
    return wrap_df(ldf.collect(engine, callback))
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
po

In [26]:
import polars as pl
import os
import gc

# 경로 설정
source_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
output_path = "/home/tracerofjageum/aml_features_final_advanced.parquet"

print("--- [최종 공정] 타입 방어 및 전수 피처 통합 시작 ---")

try:
    # 1. 원본 데이터 로드
    df = pl.read_parquet(source_path)
    
    # [타입 방어 1] Timestamp 처리 (숫자/문자 모두 대응)
    if df["Timestamp"].dtype in [pl.Int64, pl.Float64]:
        df = df.with_columns(pl.from_epoch("Timestamp", time_unit="s"))
    else:
        df = df.with_columns(pl.col("Timestamp").cast(pl.Utf8).str.to_datetime())

    # [타입 방어 2] 모든 분석 대상 컬럼 타입 통일
    df = df.with_columns([
        pl.col("Amount Paid").cast(pl.Float64),
        pl.col("Amount Received").cast(pl.Float64),
        pl.col("From Bank").cast(pl.Utf8),
        pl.col("To Bank").cast(pl.Utf8),
        pl.col("from_acc").cast(pl.Utf8),
        pl.col("to_acc").cast(pl.Utf8)
    ]).sort("Timestamp")

    # -------------------------------------------------------
    # 2. 고위험 국가 매핑 (규빈님 팀 로직 반영)
    # -------------------------------------------------------
    risk_keywords = ["Russia", "Russian", "Mexico", "Saudi", "China", "Crypto", "Digital", "Iran", "Korea"]
    risk_pattern = "(?i)" + "|".join(risk_keywords)
    
    df = df.with_columns([
        pl.col("From Bank").str.contains(risk_pattern).alias("_from_risk"),
        pl.col("To Bank").str.contains(risk_pattern).alias("_to_risk"),
        (pl.col("From Bank") != pl.col("To Bank")).alias("_is_cross_border")
    ])

    # -------------------------------------------------------
    # 3. 관점 변환 (Stacking: Edge -> Node)
    # -------------------------------------------------------
    print("🔄 데이터 관점 변환 중...")
    base_cols = ["Timestamp", "Payment Format", "Is Laundering", "src_entity_id", "_is_cross_border", "Is_High_Risk_Format"]
    
    df_out = df.select([
        *base_cols,
        pl.col("from_acc").alias("account_id"),
        pl.col("to_acc").alias("counterparty"),
        pl.lit(1).cast(pl.UInt8).alias("_is_out"),
        pl.col("Amount Paid").alias("_amt"),
        pl.col("_from_risk").alias("_is_risk")
    ])
    
    df_in = df.select([
        *base_cols,
        pl.col("to_acc").alias("account_id"),
        pl.col("from_acc").alias("counterparty"),
        pl.lit(0).cast(pl.UInt8).alias("_is_out"),
        pl.col("Amount Received").alias("_amt"),
        pl.col("_to_risk").alias("_is_risk")
    ])
    
    df_stacked = pl.concat([df_out, df_in])
    del df, df_out, df_in
    gc.collect()

    # -------------------------------------------------------
    # 4. 사전 연산 (요일, 벤포드, 시간차)
    # -------------------------------------------------------
    print("🔧 초정밀 피처 사전 연산 중...")
    df_stacked = df_stacked.with_columns([
        pl.col("Timestamp").dt.truncate("1h").alias("time_group"),
        pl.col("Timestamp").dt.weekday().alias("weekday"),
        pl.col("_amt").cast(pl.Utf8).str.slice(0, 1).alias("_first_digit"),
        (pl.col("Timestamp").diff().dt.total_seconds()).over("account_id").fill_null(0).alias("_time_delta")
    ])

    # -------------------------------------------------------
    # 5. 1시간 단위 고도화 집계 (표의 로직 100% 반영)
    # -------------------------------------------------------
    print("🔄 고도화 집계 수행 중...")
    df_agg = df_stacked.group_by(["account_id", "time_group"]).agg([
        pl.len().alias("cnt_1h"),
        pl.col("_time_delta").mean().alias("time_delta_mean"),
        pl.col("_time_delta").std().fill_null(0).alias("time_delta_std"),
        pl.col("_time_delta").min().alias("min_inter_tx_gap"),
        (pl.col("Timestamp").dt.hour() < 6).sum().alias("cnt_night"),
        (pl.col("weekday") >= 6).sum().alias("cnt_weekend"),
        pl.col("_amt").sum().alias("sum_1h"),
        pl.col("_amt").max().alias("max_1h"),
        pl.col("_amt").std().fill_null(0).alias("std_1h"),
        pl.col("_amt").filter(pl.col("_is_out") == 1).sum().alias("sum_out_1h"),
        pl.col("_amt").filter(pl.col("_is_out") == 0).sum().alias("sum_in_1h"),
        (pl.col("_amt") < 10000).sum().alias("cnt_small_tx"),
        pl.col("_is_risk").sum().alias("cnt_risk_country"),
        pl.col("_is_cross_border").mean().alias("ratio_cross_border"),
        pl.col("counterparty").n_unique().alias("degree_1h"),
        pl.col("src_entity_id").n_unique().alias("entity_acct_cnt"),
        pl.col("Is_High_Risk_Format").sum().alias("cnt_risk_format"),
        (pl.col("_first_digit") == "1").mean().alias("benford_1_ratio"),
        (pl.col("_amt") + 1).log().mean().alias("avg_log_amount"),
        pl.col("Is Laundering").max().alias("is_laundering")
    ])

    # -------------------------------------------------------
    # 6. 장기 패턴 및 최종 업그레이드 지표 산출
    # -------------------------------------------------------
    print("🌊 장기 패턴 및 최종 지표 산출 중...")
    df_final = df_agg.sort(["account_id", "time_group"]).with_columns([
        pl.col("sum_1h").cum_sum().over("account_id").alias("_sum_cum"),
        pl.col("cnt_1h").cum_sum().over("account_id").alias("_cnt_cum")
    ]).with_columns([
        (pl.col("_sum_cum").shift(1) - pl.col("_sum_cum").shift(25)).over("account_id").fill_null(0).alias("sum_24h"),
        (pl.col("_cnt_cum").shift(1) - pl.col("_cnt_cum").shift(25)).over("account_id").fill_null(0).alias("cnt_24h"),
        (pl.col("_sum_cum").shift(1) - pl.col("_sum_cum").shift(169)).over("account_id").fill_null(0).alias("_sum_7d")
    ]).with_columns([
        (pl.col("_sum_7d") / 168).alias("avg_7d"),
        (pl.col("cnt_1h") / (pl.col("cnt_24h") / 24 + 1)).alias("burst_ratio"),
        (pl.col("sum_in_1h") / (pl.col("sum_out_1h") + 1)).alias("in_out_balance_ratio"),
        (pl.col("time_delta_std") / (pl.col("time_delta_mean") + 1e-6)).alias("inter_arrival_cv"),
        (pl.col("benford_1_ratio") - 0.301).abs().alias("benford_deviation"),
        (pl.col("cnt_risk_country") / (pl.col("cnt_1h") + 1)).alias("ratio_risk_country")
    ])

    final_cols = [c for c in df_final.columns if not c.startswith("_")]
    df_final.select(final_cols).write_parquet(output_path)
    
    print("-" * 30)
    print(f"✅ [미션 완료] 모든 업그레이드 피처 생성 완료: {output_path}")
    print(f"📊 컬럼 수: {len(df_final.select(final_cols).columns)} / 행 수: {len(df_final):,}")
    print("-" * 30)

except Exception as e:
    print(f"❌ 에러 발생: {e}")
    import traceback
    traceback.print_exc()

--- [최종 공정] 타입 방어 및 전수 피처 통합 시작 ---
🔄 데이터 관점 변환 중...
❌ 에러 발생: unable to find column "Is_High_Risk_Format"; valid columns: ["Timestamp", "From Bank", "from_acc", "To Bank", "to_acc", "Amount Received", "Receiving Currency", "Amount Paid", "Payment Currency", "Payment Format", "Is Laundering", "src_bank", "Bank ID", "src_entity_id", "src_entity", "dst_bank", "Bank ID_right", "dst_entity_id", "dst_entity", "_from_risk", "_to_risk", "_is_cross_border"]


Traceback (most recent call last):
  File "/tmp/ipykernel_29942/1054276019.py", line 49, in <module>
    df_out = df.select([
             ^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/dataframe/frame.py", line 10307, in select
    .collect(optimizations=QueryOptFlags._eager())
     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/_utils/deprecation.py", line 97, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/lazyframe/opt_flags.py", line 326, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/polars/lazyframe/frame.py", line 2440, in collect
    return wrap_df(ldf.collect(engine, callback))
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
polars.excep

In [27]:
import polars as pl
import os
import gc

# 경로 설정
source_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
output_path = "/home/tracerofjageum/aml_features_final_advanced.parquet"

print("--- [최종 공정] 결제 수단 로직 추가 및 전수 피처 통합 시작 ---")

try:
    # 1. 원본 데이터 로드
    df = pl.read_parquet(source_path)
    
    # [타입 방어] Timestamp 및 기본 컬럼 타입 통일
    if df["Timestamp"].dtype in [pl.Int64, pl.Float64]:
        df = df.with_columns(pl.from_epoch("Timestamp", time_unit="s"))
    else:
        df = df.with_columns(pl.col("Timestamp").cast(pl.Utf8).str.to_datetime())

    df = df.with_columns([
        pl.col("Amount Paid").cast(pl.Float64),
        pl.col("Amount Received").cast(pl.Float64),
        pl.col("From Bank").cast(pl.Utf8),
        pl.col("To Bank").cast(pl.Utf8),
        pl.col("from_acc").cast(pl.Utf8),
        pl.col("to_acc").cast(pl.Utf8),
        pl.col("Payment Format").cast(pl.Utf8)
    ]).sort("Timestamp")

    # -------------------------------------------------------
    # 2. 고위험 요소 매핑 (국가 + 결제 수단)
    # -------------------------------------------------------
    risk_countries = ["Russia", "Russian", "Mexico", "Saudi", "China", "Crypto", "Digital", "Iran", "Korea"]
    # 추적이 어려운 고위험 결제 수단 정의
    high_risk_formats = ["Cash", "Cheque", "Crypto", "Bitcoin"] 
    
    df = df.with_columns([
        pl.col("From Bank").str.contains("(?i)" + "|".join(risk_countries)).alias("_from_risk"),
        pl.col("To Bank").str.contains("(?i)" + "|".join(risk_countries)).alias("_to_risk"),
        pl.col("Payment Format").str.contains("(?i)" + "|".join(high_risk_formats)).alias("_is_high_risk_format"),
        (pl.col("From Bank") != pl.col("To Bank")).alias("_is_cross_border")
    ])

    # -------------------------------------------------------
    # 3. 관점 변환 (Stacking: Edge -> Node)
    # -------------------------------------------------------
    print("🔄 데이터 관점 변환 중...")
    # 존재하는 컬럼만 선택
    base_cols = ["Timestamp", "Is Laundering", "src_entity_id", "_is_cross_border", "_is_high_risk_format"]
    
    df_out = df.select([
        *base_cols,
        pl.col("from_acc").alias("account_id"),
        pl.col("to_acc").alias("counterparty"),
        pl.lit(1).cast(pl.UInt8).alias("_is_out"),
        pl.col("Amount Paid").alias("_amt"),
        pl.col("_from_risk").alias("_is_risk")
    ])
    
    df_in = df.select([
        *base_cols,
        pl.col("to_acc").alias("account_id"),
        pl.col("from_acc").alias("counterparty"),
        pl.lit(0).cast(pl.UInt8).alias("_is_out"),
        pl.col("Amount Received").alias("_amt"),
        pl.col("_to_risk").alias("_is_risk")
    ])
    
    df_stacked = pl.concat([df_out, df_in])
    del df, df_out, df_in
    gc.collect()

    # -------------------------------------------------------
    # 4. 사전 연산 (요일, 벤포드, 시간차)
    # -------------------------------------------------------
    print("🔧 초정밀 피처 사전 연산 중...")
    df_stacked = df_stacked.with_columns([
        pl.col("Timestamp").dt.truncate("1h").alias("time_group"),
        pl.col("Timestamp").dt.weekday().alias("weekday"),
        pl.col("_amt").cast(pl.Utf8).str.slice(0, 1).alias("_first_digit"),
        (pl.col("Timestamp").diff().dt.total_seconds()).over("account_id").fill_null(0).alias("_time_delta")
    ])

    # -------------------------------------------------------
    # 5. 1시간 단위 고도화 집계 (표의 로직 100% 반영)
    # -------------------------------------------------------
    print("🔄 고도화 집계 수행 중 (Structuring, Pass-through, Layering)...")
    df_agg = df_stacked.group_by(["account_id", "time_group"]).agg([
        pl.len().alias("cnt_1h"),
        pl.col("_time_delta").mean().alias("time_delta_mean"),
        pl.col("_time_delta").std().fill_null(0).alias("time_delta_std"),
        pl.col("_time_delta").min().alias("min_inter_tx_gap"),
        (pl.col("Timestamp").dt.hour() < 6).sum().alias("cnt_night"),
        (pl.col("weekday") >= 6).sum().alias("cnt_weekend"),
        pl.col("_amt").sum().alias("sum_1h"),
        pl.col("_amt").max().alias("max_1h"),
        pl.col("_amt").std().fill_null(0).alias("std_1h"),
        pl.col("_amt").filter(pl.col("_is_out") == 1).sum().alias("sum_out_1h"),
        pl.col("_amt").filter(pl.col("_is_out") == 0).sum().alias("sum_in_1h"),
        (pl.col("_amt") < 10000).sum().alias("cnt_small_tx"),
        pl.col("_is_risk").sum().alias("cnt_risk_country"),
        pl.col("_is_cross_border").mean().alias("ratio_cross_border"),
        pl.col("counterparty").n_unique().alias("degree_1h"),
        pl.col("src_entity_id").n_unique().alias("entity_acct_cnt"),
        pl.col("_is_high_risk_format").sum().alias("cnt_risk_format"),
        (pl.col("_first_digit") == "1").mean().alias("benford_1_ratio"),
        (pl.col("_amt") + 1).log().mean().alias("avg_log_amount"),
        pl.col("Is Laundering").max().alias("is_laundering")
    ])

    # -------------------------------------------------------
    # 6. 장기 패턴 및 최종 업그레이드 지표 산출
    # -------------------------------------------------------
    print("🌊 장기 패턴 및 최종 지표 산출 중...")
    df_final = df_agg.sort(["account_id", "time_group"]).with_columns([
        pl.col("sum_1h").cum_sum().over("account_id").alias("_sum_cum"),
        pl.col("cnt_1h").cum_sum().over("account_id").alias("_cnt_cum")
    ]).with_columns([
        (pl.col("_sum_cum").shift(1) - pl.col("_sum_cum").shift(25)).over("account_id").fill_null(0).alias("sum_24h"),
        (pl.col("_cnt_cum").shift(1) - pl.col("_cnt_cum").shift(25)).over("account_id").fill_null(0).alias("cnt_24h"),
        (pl.col("_sum_cum").shift(1) - pl.col("_sum_cum").shift(169)).over("account_id").fill_null(0).alias("_sum_7d")
    ]).with_columns([
        (pl.col("_sum_7d") / 168).alias("avg_7d"),
        (pl.col("cnt_1h") / (pl.col("cnt_24h") / 24 + 1)).alias("burst_ratio"),
        (pl.col("sum_in_1h") / (pl.col("sum_out_1h") + 1)).alias("in_out_balance_ratio"),
        (pl.col("time_delta_std") / (pl.col("time_delta_mean") + 1e-6)).alias("inter_arrival_cv"),
        (pl.col("benford_1_ratio") - 0.301).abs().alias("benford_deviation"),
        (pl.col("cnt_risk_country") / (pl.col("cnt_1h") + 1)).alias("ratio_risk_country")
    ])

    final_cols = [c for c in df_final.columns if not c.startswith("_")]
    df_final.select(final_cols).write_parquet(output_path)
    
    print("-" * 30)
    print(f"✅ [최종 완료] 모든 피처 반영 완료: {output_path}")
    print(f"📊 컬럼 수: {len(df_final.select(final_cols).columns)} / 행 수: {len(df_final):,}")
    print("-" * 30)

except Exception as e:
    print(f"❌ 에러 발생: {e}")
    import traceback
    traceback.print_exc()

    

--- [최종 공정] 결제 수단 로직 추가 및 전수 피처 통합 시작 ---
🔄 데이터 관점 변환 중...
🔧 초정밀 피처 사전 연산 중...
🔄 고도화 집계 수행 중 (Structuring, Pass-through, Layering)...
🌊 장기 패턴 및 최종 지표 산출 중...
------------------------------
✅ [최종 완료] 모든 피처 반영 완료: /home/tracerofjageum/aml_features_final_advanced.parquet
📊 컬럼 수: 30 / 행 수: 28,756,304
------------------------------


In [28]:
import polars as pl

# 1. 데이터 로드
output_path = "/home/tracerofjageum/aml_features_final_advanced.parquet"
df = pl.read_parquet(output_path)

# 2. 전체 컬럼 목록 및 타입 확인
print(f"✅ 전체 데이터 Shape: {df.shape}")
print("-" * 50)
print(f"{'컬럼명':<25} | {'데이터 타입':<15} | {'샘플 값'}")
print("-" * 50)

sample_row = df.head(1).to_dicts()[0]
for col in df.columns:
    dtype = str(df.schema[col])
    val = sample_row[col]
    print(f"{col:<25} | {dtype:<15} | {val}")

print("-" * 50)

# 3. 카테고리별 핵심 피처 작동 여부 확인
print("\n[🎯 업그레이드 피처 작동 테스트]")
test_cases = {
    "Structuring": ["cnt_small_tx", "burst_ratio"],
    "Pass-through": ["min_inter_tx_gap", "in_out_balance_ratio", "inter_arrival_cv"],
    "Layering": ["entity_acct_cnt", "ratio_cross_border", "benford_deviation"],
    "High Risk": ["ratio_risk_country", "cnt_risk_format"]
}

for category, cols in test_cases.items():
    available = [c for c in cols if c in df.columns]
    print(f"🔹 {category:<12}: {available} ✅")

# 4. 타겟 라벨(자금세탁) 재확인
print("\n[🚨 타겟 라벨 분포]")
print(df["is_laundering"].value_counts())

✅ 전체 데이터 Shape: (28756304, 30)
--------------------------------------------------
컬럼명                       | 데이터 타입          | 샘플 값
--------------------------------------------------
account_id                | String          | 100428660
time_group                | Datetime(time_unit='us', time_zone=None) | 2022-09-01 00:00:00
cnt_1h                    | UInt32          | 10925
time_delta_mean           | Float64         | -125.98627002288329
time_delta_std            | Float64         | 13223.544137720522
min_inter_tx_gap          | Int64           | -1382160
cnt_night                 | UInt32          | 10925
cnt_weekend               | UInt32          | 0
sum_1h                    | Float64         | 27299711092.56609
max_1h                    | Float64         | 4351720448.0
std_1h                    | Float64         | 61069748.74539183
sum_out_1h                | Float64         | 27299711091.32609
sum_in_1h                 | Float64         | 1.24
cnt_small_tx              | U